# Notebook Overview: Rare Disease Candidate Ranking Workflow

This notebook builds a local rare-disease ranking pipeline for patient case reports. It combines dense embedding similarity, UMLS concept extraction, BM25-based sparse scoring, reciprocal-rank fusion, and a final Qwen reranker that scores disease knowledge sections against each patient case.

## Main Goal

Given patient-level JSON files containing `patient_id`, `dataset`, and `case_data`, the notebook produces ranked candidate diseases using multiple evidence layers:

1. Dense embedding similarity between patient case text and disease knowledge.
2. UMLS concept extraction from case reports using SciSpaCy.
3. Sparse BM25 matching between patient UMLS concepts and disease-level UMLS/BM25 knowledge.
4. Dense + sparse fusion using Reciprocal Rank Fusion (RRF).
5. Qwen reranking of the fused candidate diseases using LLM-generated disease knowledge sections.

## Core Inputs

| Input | Purpose |
|---|---|
| `/base_dir/test_pts/*.json` | Patient case-report JSON files used by the embedding reranker and UMLS extraction steps. |
| `/base_dir/utilities/umls_mapping.xlsx` | Maps UMLS semantic type IDs to readable semantic type names. |
| `/base_dir/utilities/claude_BM25.xlsx` | Disease-level UMLS/BM25 knowledge table used for sparse disease scoring. |
| `/base_dir/utilities/claude_sections.xlsx` | LLM-generated disease knowledge sections used to build documents for the Qwen reranker. |
| `/base_dir/combined_test_pts.xlsx` | Patient case data table used when generating RRF-based reranker JSON files. |
| Local cached embedding checkpoint | `r76941156/rare-disease-embedding-model`, loaded from the local Hugging Face cache snapshot. |
| Qwen reranker model | `Qwen/Qwen3-Reranker-8B`, cached under `/base_dir/hf_model_cache`. |

## Workflow Summary

### 1. Dense embedding reranking

The notebook first loads the rare-disease embedding model from the local Hugging Face cache and embeds each patient case together with disease documents. It uses cosine similarity and writes patient-level reranked outputs.

**Main output:**

```text
/base_dir/dataset_json_output_claude
```

### 2. Extract UMLS concepts from patient cases

The notebook loads `en_core_sci_scibert` with the SciSpaCy UMLS linker and extracts UMLS concepts from each patient case report. Long notes are split into token-safe chunks before entity linking.

**Main output pattern:**

```text
/base_dir/dataset_json_output_claude/{dataset}/{patient_id}/*_case_data_umls_concepts.csv
```

### 3. Filter patient UMLS concepts

Extracted UMLS concepts are filtered to keep clinically relevant semantic types, remove low-confidence matches, and exclude negated concept names that start with terms such as `not`, `no`, or `without`.

**Main output:**

```text
/base_dir/dataset_json_output_claude_filtered
```

### 4. Calculate BM25-based disease scores

The notebook joins patient UMLS concepts with the disease-level BM25 knowledge table and aggregates shared UMLS evidence per disease. This produces patient-level candidate disease lists from sparse concept matching.

**Main output:**

```text
/base_dir/patient_top10_diseases_bm25_local
```

### 5. Fuse dense and sparse ranking results with RRF

Dense embedding results and BM25 sparse results are merged by patient and disease name. The notebook calculates an RRF score using `k_rrf = 60`, then writes fused candidate disease rankings.

**Main output:**

```text
/base_dir/fusion_dense_sparse_rrf_local
```

### 6. Build Qwen reranker input JSON files

For each fused patient result, the notebook pulls disease knowledge sections from `claude_sections.xlsx` and combines them with patient `case_data`. The resulting JSON files become the input documents for the final reranker.

**Main output:**

```text
/base_dir/rrf_top10_patient_jsons/claude
```

### 7. Run Qwen reranker on fused candidates

The notebook loads `Qwen/Qwen3-Reranker-8B`, formats each patient case as the query and each disease knowledge profile as the document, then scores whether the disease explains the patient condition. Final outputs are saved as JSON-formatted `.txt` files with `detailed_scores`.

**Main output:**

```text
/base_dir/qwen_reranker_rrf_top10_output/claude
```

## Key Final Output Structure

The final Qwen reranker output contains:

```text
patient_id, dataset, model, source_json, case_data and detailed_scores
```

Each record in `detailed_scores` contains:
```text
the original RRF rank, disease name, reranker score, and final Qwen rerank position.
```

## Practical Notes

- Both the embedding and reranker sections are configured to use **CPU with `torch.float32`** for local Mac runs. The embedding step uses CPU to avoid NaN issues observed with Mac MPS and `float16` disease embeddings, while the Qwen reranker also uses CPU because Mac MPS caused out-of-memory errors during local reranking.
- The notebook assumes a local folder structure under `/Users/sandersu1/downloads`. Update `base_dir` to match your own local environment before running the workflow.
- Most sections currently use `model_tag = "claude"` or `models = ["claude"]`; these can be expanded for other models if the corresponding files exist.

- The final reranker step can be slow because it loads an 8B Qwen reranker and scores patient-disease document pairs locally.
- For a better user experience, especially for full-scale runs, using at least 1 H100 GPU is recommended.
- The first run may take additional time because the program may need to download both the embedding model and the reranker model from Hugging Face. After the models are cached locally, later runs should start faster.


## Qwen3-8B FT Embedding Ranker (with Claude generated disease knowledge)
### Huggingface Repo: https://huggingface.co/r76941156/rare-disease-embedding-model

In [3]:
import pandas as pd
import os
import re
import glob
import json
import numpy as np
from tqdm import tqdm
import time
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from huggingface_hub import snapshot_download

In [4]:
# ============================================================
# Config
# ============================================================


# Model / repo settings
model_tag = "claude"
repo_id = "r76941156/rare-disease-embedding-model"

# Base folder
base_dir = "/Users/sandersu1/downloads"

# Input / output folders
input_root = os.path.join(
    base_dir,
    "test_pts"
)

output_root = os.path.join(
    base_dir,
    f"dataset_json_output_{model_tag}"
)

# Optional: create output folder
os.makedirs(output_root, exist_ok=True)

print(f"Model tag: {model_tag}")
print(f"Repo ID: {repo_id}")
print(f"Input root: {input_root}")
print(f"Output root: {output_root}")

Model tag: claude
Repo ID: r76941156/rare-disease-embedding-model
Input root: /Users/sandersu1/downloads/test_pts
Output root: /Users/sandersu1/downloads/dataset_json_output_claude


In [5]:

# ============================================================
# 0. Config
# ============================================================

# First clean run: True
# After confirming no NaN, you can change to False
REBUILD_DISEASE_CACHE = False

# If True, rerun even if output txt already exists
FORCE_OVERWRITE = True

task_instruction = (
    "Given a patient case description, retrieve relevant rare diseases that explain the clinical findings"
)


# ============================================================
# 1. Load cached Hugging Face snapshot
# ============================================================

checkpoint_dir = snapshot_download(
    repo_id=repo_id,
    repo_type="model",
    local_files_only=True
)

print("Using cached model snapshot:")
print(checkpoint_dir)


# ============================================================
# 2. Device setup for Mac
# ============================================================


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

# Important:
# Mac MPS + float16 caused NaN in disease embeddings.
# Use float32 on Mac MPS and CPU.
# Only CUDA uses float16.
if device.type == "cuda":
    torch_dtype = torch.float16
else:
    torch_dtype = torch.float32

print("Using torch dtype:", torch_dtype)


# ============================================================
# 3. Load tokenizer and model
# ============================================================

print("🔄 Loading local cached rare disease embedding model ...")

tokenizer = AutoTokenizer.from_pretrained(
    checkpoint_dir,
    padding_side="left",
    local_files_only=True
)

model = AutoModel.from_pretrained(
    checkpoint_dir,
    torch_dtype=torch_dtype,
    local_files_only=True,
    low_cpu_mem_usage=True
).to(device)

model.eval()

print("✅ Model loaded successfully.")


# ============================================================
# 4. Pooling strategy: last token pooling
# ============================================================

def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]

    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        return last_hidden_states[
            torch.arange(last_hidden_states.size(0), device=last_hidden_states.device),
            sequence_lengths
        ]


# ============================================================
# 5. Embedding function
# ============================================================

def encode_texts(texts, is_query: bool, batch_size: int = 8) -> Tensor:
    """
    Encode texts into normalized embeddings.

    For Mac:
    - Use smaller batch_size to reduce memory pressure.
    - Convert pooled vectors to float32 before normalization.
    """

    if is_query:
        texts = [
            f"Instruct: {task_instruction}\nQuery: {t}"
            for t in texts
        ]

    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        tokens = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        tokens = {k: v.to(device) for k, v in tokens.items()}

        with torch.no_grad():
            outputs = model(**tokens)
            pooled = last_token_pool(
                outputs.last_hidden_state,
                tokens["attention_mask"]
            )

            # Important for numerical stability
            pooled = pooled.float()
            embeddings = F.normalize(pooled, p=2, dim=1)

        all_embeddings.append(embeddings.cpu())

    final_embeddings = torch.cat(all_embeddings, dim=0).to(device)

    return final_embeddings


# ============================================================
# 6. Disease embedding cache
# ============================================================

disease_cache_path = os.path.join(output_root, "cached_disease_embeddings.npy")
disease_name_cache_path = os.path.join(output_root, "cached_disease_names.json")

if REBUILD_DISEASE_CACHE:
    for p in [disease_cache_path, disease_name_cache_path]:
        if os.path.exists(p):
            os.remove(p)
            print(f"Removed old cache: {p}")

global_cached_embeds = None
global_cached_diseases = None


def extract_disease_texts_and_names(diseases):
    disease_texts = []
    disease_names = []

    for d in diseases:
        name = d.get("disease", "").strip()

        # Keep your current ablation: disease name only
        full_text = (
            f"{name}\n"
            f"### Clinical Presentation:\n{d.get('clinical_presentation_section', '')}\n"
            f"### Diagnostic Evaluation:\n{d.get('diagnostic_evaluation_section', '')}\n"
            f"### Subtype Variant:\n{d.get('subtype_variant_section', '')}\n"
            f"### Management Therapy:\n{d.get('management_therapy_section', '')}"
        ).strip()

        if name:
            disease_texts.append(full_text)
            disease_names.append(name)

    return disease_texts, disease_names


def build_or_load_disease_cache(diseases):
    global global_cached_embeds, global_cached_diseases

    disease_texts, disease_names = extract_disease_texts_and_names(diseases)

    if not disease_texts:
        raise ValueError("No valid disease names found in input diseases.")

    # Reuse in-memory cache only if disease list is identical
    if global_cached_embeds is not None and global_cached_diseases == disease_names:
        return global_cached_embeds, global_cached_diseases

    # Reset memory cache if disease list changes
    if global_cached_embeds is not None and global_cached_diseases != disease_names:
        print("⚠️ Disease list changed. Rebuilding in-memory disease cache.")
        global_cached_embeds = None
        global_cached_diseases = None

    use_cache = False

    if os.path.exists(disease_cache_path) and os.path.exists(disease_name_cache_path):
        print("📦 Checking existing disease embedding cache ...")

        arr = np.load(disease_cache_path)

        with open(disease_name_cache_path, "r", encoding="utf-8") as f:
            cached_names = json.load(f)

        cache_valid = (
            arr.shape[0] == len(disease_names)
            and len(cached_names) == len(disease_names)
            and cached_names == disease_names
            and np.isfinite(arr).all()
        )

        if cache_valid:
            use_cache = True
        else:
            print("⚠️ Existing disease cache is invalid and will be rebuilt.")
            print("Cache embedding shape:", arr.shape)
            print("Expected diseases:", len(disease_names))
            print("NaN count:", np.isnan(arr).sum())
            print("Inf count:", np.isinf(arr).sum())

            for p in [disease_cache_path, disease_name_cache_path]:
                if os.path.exists(p):
                    os.remove(p)
                    print(f"Removed invalid cache: {p}")

    if use_cache:
        print("📦 Loading valid cached disease embeddings ...")
        arr = np.load(disease_cache_path)
        global_cached_embeds = torch.tensor(arr, dtype=torch.float32).to(device)
        global_cached_diseases = disease_names

    else:
        print("⚙️ Building disease embedding cache ...")
        doc_embs = encode_texts(
            disease_texts,
            is_query=False,
            batch_size=8
        )

        if torch.isnan(doc_embs).any():
            raise ValueError("Newly generated disease embeddings contain NaN.")

        if torch.isinf(doc_embs).any():
            raise ValueError("Newly generated disease embeddings contain Inf.")

        np.save(disease_cache_path, doc_embs.cpu().float().numpy())

        with open(disease_name_cache_path, "w", encoding="utf-8") as f:
            json.dump(disease_names, f, ensure_ascii=False, indent=2)

        global_cached_embeds = doc_embs.float().to(device)
        global_cached_diseases = disease_names

    print("Disease cache ready.")
    print("Number of diseases:", len(global_cached_diseases))
    print("Disease embedding shape:", global_cached_embeds.shape)
    print("Disease embeddings has NaN:", torch.isnan(global_cached_embeds).any().item())

    return global_cached_embeds, global_cached_diseases


# ============================================================
# 7. Reranking logic
# ============================================================

def rerank_by_embedding(case_text, diseases, top_k=None):
    if not diseases or not case_text.strip():
        return []

    doc_embs, disease_names = build_or_load_disease_cache(diseases)

    query_emb = encode_texts(
        [case_text],
        is_query=True,
        batch_size=1
    )

    if torch.isnan(doc_embs).any():
        raise ValueError("doc_embs contains NaN.")

    if torch.isnan(query_emb).any():
        raise ValueError("query_emb contains NaN.")

    scores = torch.matmul(
        query_emb.float(),
        doc_embs.float().T
    ).squeeze(0).cpu().numpy()

    if np.isnan(scores).any():
        nan_idx = np.where(np.isnan(scores))[0]
        print("NaN disease examples:")
        for idx in nan_idx[:10]:
            print(idx, disease_names[idx])

        raise ValueError("Scores contain NaN. Disease embeddings are invalid.")

    # Important:
    # Do NOT filter by if s > 0.
    # Keep all diseases and rank them by score.
    ranked = [
        {
            "disease": disease_names[i],
            "match_score": round(float(s), 4),
            "original_rank": i + 1
        }
        for i, s in enumerate(scores)
    ]

    ranked.sort(key=lambda x: -x["match_score"])

    for i, entry in enumerate(ranked, start=1):
        entry["rerank_position"] = i

    return ranked[:top_k] if top_k else ranked


# ============================================================
# 8. Process one patient JSON file
# ============================================================

def process_json_file(json_path):
    base_name = os.path.splitext(os.path.basename(json_path))[0]
    output_txt = os.path.join(output_root, f"{base_name}_cosine_reranked.txt")

    if os.path.exists(output_txt) and os.path.getsize(output_txt) > 0 and not FORCE_OVERWRITE:
        print(f"⏭️ Skipping {json_path} (already processed)")
        return

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            entry = json.load(f)

        patient_id = entry.get("patient_id", base_name)
        case_text = entry.get("case_data", "")
        diseases = entry.get("diseases", [])

        reranked = rerank_by_embedding(case_text, diseases)

        with open(output_txt, "w", encoding="utf-8") as out_f:
            json.dump(
                {
                    "patient_id": patient_id,
                    "reranked": reranked
                },
                out_f,
                indent=2
            )

        print(f"✅ Reranked: {output_txt} ({len(reranked)} kept, {len(diseases)} total)")

    except Exception as e:
        print(f"❌ Error in {json_path}: {e}")

        with open(output_txt, "w", encoding="utf-8") as out_f:
            json.dump(
                {
                    "patient_id": os.path.splitext(os.path.basename(json_path))[0],
                    "error": str(e)
                },
                out_f,
                indent=2
            )


# ============================================================
# 9. Main driver
# ============================================================

if __name__ == "__main__":
    all_json_files = sorted([
        f for f in os.listdir(input_root)
        if f.endswith(".json")
    ])

    print(f"📂 Found {len(all_json_files)} JSON files in {input_root}")

    for json_file in tqdm(all_json_files, desc="📊 Reranking with Instruction", unit="file"):
        json_path = os.path.join(input_root, json_file)
        #process_json_file(json_path)

        start_time = time.perf_counter()
        process_json_file(json_path)
        end_time = time.perf_counter()

        print(f"⏱️ Running time for {json_file}: {end_time - start_time:.2f} seconds")

    print(f"\n🎉 All done! Results saved to: {output_root}")

Using cached model snapshot:
/Users/sandersu1/.cache/huggingface/hub/models--r76941156--rare-disease-embedding-model/snapshots/28c67d01a89a923b3f0d5e3e69757b7e3614c69e
Using device: cpu
Using torch dtype: torch.float32
🔄 Loading local cached rare disease embedding model ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Model loaded successfully.
📂 Found 10 JSON files in /Users/sandersu1/downloads/test_pts


📊 Reranking with Instruction:   0%|                   | 0/10 [00:00<?, ?file/s]

📦 Checking existing disease embedding cache ...
📦 Loading valid cached disease embeddings ...
Disease cache ready.
Number of diseases: 1320
Disease embedding shape: torch.Size([1320, 4096])
Disease embeddings has NaN: False


...

✅ Reranked: /Users/sandersu1/downloads/dataset_json_output_claude/6935327-1_pubmed_cosine_reranked.txt (1320 kept, 1320 total)
⏱️ Running time for 6935327-1_pubmed.json: 4.41 seconds


📊 Reranking with Instruction:  90%|█████████▉ | 9/10 [00:41<00:04,  4.60s/file]

✅ Reranked: /Users/sandersu1/downloads/dataset_json_output_claude/7007742-1_pubmed_cosine_reranked.txt (1320 kept, 1320 total)
⏱️ Running time for 7007742-1_pubmed.json: 6.50 seconds


📊 Reranking with Instruction: 100%|██████████| 10/10 [00:43<00:00,  4.33s/file]

✅ Reranked: /Users/sandersu1/downloads/dataset_json_output_claude/7058835-1_pubmed_cosine_reranked.txt (1320 kept, 1320 total)
⏱️ Running time for 7058835-1_pubmed.json: 2.14 seconds

🎉 All done! Results saved to: /Users/sandersu1/downloads/dataset_json_output_claude


## Save top-10 predicted diseases for each patient for later RRF use

In [7]:

# ============================================================
# Settings
# ============================================================

# Existing embedding reranker output folder
# This should be the folder where files like *_cosine_reranked.txt are saved
embedding_output_root = os.path.join(
    base_dir,
    "dataset_json_output_claude"
)

# Optional: original input JSON folder, used to recover dataset if output txt does not include dataset
# If you do not need it, leave it as None
input_json_root = os.path.join(
    base_dir,
    "test_pts"
)

# New folder to save top10 output per patient
top10_output_root = os.path.join(
    base_dir,
    "embedding_reranker_top10_local"
)

# New folder to save evaluation
eval_output_root = os.path.join(
    base_dir,
    "embedding_reranker_top10_eval"
)

os.makedirs(top10_output_root, exist_ok=True)
os.makedirs(eval_output_root, exist_ok=True)

top_n = 10


# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    x = str(x)
    x = re.sub(r"[^\w\-\.]+", "_", x)
    return x.strip("_")


def normalize_text(x):
    if pd.isna(x):
        return ""

    x = str(x).strip().lower()
    x = x.replace("’", "'").replace("`", "'")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x


def standardize_truth_columns(df):
    rename_map = {}

    for c in df.columns:
        c_norm = str(c).strip().lower()

        if c_norm == "dataset":
            rename_map[c] = "dataset"

        elif c_norm in ["patient_id", "person_id", "pt_id"]:
            rename_map[c] = "patient_id"

        elif c_norm in [
            "rare_disease_name",
            "disease",
            "correct_disease",
            "true_disease"
        ]:
            rename_map[c] = "rare_disease_name"

    df = df.rename(columns=rename_map)

    required_cols = ["dataset", "patient_id", "rare_disease_name"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"Truth file missing required columns: {missing_cols}\n"
            f"Current columns: {list(df.columns)}"
        )

    df = df[required_cols].copy()

    df["dataset"] = df["dataset"].astype(str).str.strip()
    df["patient_id"] = df["patient_id"].astype(str).str.strip()
    df["rare_disease_name"] = df["rare_disease_name"].astype(str).str.strip()
    df["true_disease_norm"] = df["rare_disease_name"].apply(normalize_text)

    df = df.drop_duplicates().reset_index(drop=True)

    return df


def parse_dataset_from_filename(path):
    """
    Try to infer dataset from filename.

    Examples:
    5941722-1_pubmed_cosine_reranked.txt -> pubmed
    3884936-1_pubmed_cosine_reranked.txt -> pubmed
    9_ramedis_cosine_reranked.txt -> ramedis
    """

    name = os.path.basename(path)
    name = name.replace("_cosine_reranked.txt", "")
    name = name.replace("_reranked.txt", "")
    name = os.path.splitext(name)[0]

    known_datasets = ["pubmed", "hms", "mme", "mygene2", "lirical", "ramedis"]

    parts = name.split("_")

    for p in parts:
        if p.lower() in known_datasets:
            return p.lower()

    return None


def parse_patient_from_filename(path):
    """
    Try to infer patient_id from filename.

    Examples:
    5941722-1_pubmed_cosine_reranked.txt -> 5941722-1
    9_ramedis_cosine_reranked.txt -> 9
    """

    name = os.path.basename(path)
    name = name.replace("_cosine_reranked.txt", "")
    name = name.replace("_reranked.txt", "")
    name = os.path.splitext(name)[0]

    known_datasets = ["pubmed", "hms", "mme", "mygene2", "lirical", "ramedis"]

    parts = name.split("_")

    if len(parts) >= 2 and parts[-1].lower() in known_datasets:
        return "_".join(parts[:-1])

    return parts[0]


def build_input_metadata_lookup(input_json_root):
    """
    Read original patient JSON files to recover:
    patient_id, dataset

    This is useful because your embedding output txt only saves patient_id,
    not always dataset.
    """

    lookup = {}

    if input_json_root is None or not os.path.exists(input_json_root):
        return lookup

    json_files = sorted(glob.glob(os.path.join(input_json_root, "*.json")))

    for json_path in json_files:
        try:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            patient_id = str(data.get("patient_id", "")).strip()
            dataset = str(data.get("dataset", "")).strip()

            if patient_id and dataset:
                lookup[patient_id] = dataset

            # also store by base filename
            base_name = os.path.splitext(os.path.basename(json_path))[0]
            if dataset:
                lookup[base_name] = dataset

        except Exception:
            continue

    return lookup


def read_embedding_rerank_file(path):
    """
    Read one *_cosine_reranked.txt file.
    """

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "error" in data:
        return None, data["error"]

    patient_id = str(data.get("patient_id", "")).strip()
    reranked = data.get("reranked", [])

    if not isinstance(reranked, list):
        reranked = []

    return {
        "patient_id": patient_id,
        "reranked": reranked
    }, None


# ============================================================
# Build metadata lookup from original input JSON
# ============================================================

input_metadata_lookup = build_input_metadata_lookup(input_json_root)

print(f"✅ Input metadata lookup records: {len(input_metadata_lookup)}")


# ============================================================
# Save top10 output for each patient
# ============================================================

rerank_files = sorted(
    glob.glob(
        os.path.join(
            embedding_output_root,
            "**",
            "*_cosine_reranked.txt"
        ),
        recursive=True
    )
)

print(f"📂 Embedding reranker output root: {embedding_output_root}")
print(f"📄 Found reranked files: {len(rerank_files)}")

top10_detail_dfs = []
top10_summary_records = []

for i, rerank_file in enumerate(rerank_files, 1):

    print("-" * 100)
    print(f"📥 [{i}/{len(rerank_files)}] Reading:")
    print(rerank_file)

    result, error = read_embedding_rerank_file(rerank_file)

    if error is not None:
        print(f"⚠️ Error file, skipping: {error}")

        top10_summary_records.append({
            "model": model_tag,
            "dataset": None,
            "patient_id": parse_patient_from_filename(rerank_file),
            "status": "error_file",
            "n_reranked": 0,
            "top10_file": "",
            "source_file": rerank_file,
            "error": error
        })

        continue

    patient_id = result["patient_id"]

    if not patient_id:
        patient_id = parse_patient_from_filename(rerank_file)

    dataset = input_metadata_lookup.get(patient_id)

    if dataset is None:
        base_name = os.path.basename(rerank_file)
        base_name = base_name.replace("_cosine_reranked.txt", "")
        dataset = input_metadata_lookup.get(base_name)

    if dataset is None:
        dataset = parse_dataset_from_filename(rerank_file)

    if dataset is None:
        dataset = "unknown_dataset"

    reranked = result["reranked"]

    if not reranked:
        print(f"⚠️ No reranked diseases for patient {patient_id}")

        top10_summary_records.append({
            "model": model_tag,
            "dataset": dataset,
            "patient_id": patient_id,
            "status": "no_reranked_diseases",
            "n_reranked": 0,
            "top10_file": "",
            "source_file": rerank_file,
            "error": ""
        })

        continue

    records = []

    for item in reranked[:top_n]:
        disease = item.get("disease", "")
        match_score = item.get("match_score", None)
        original_rank = item.get("original_rank", None)
        rerank_position = item.get("rerank_position", None)

        records.append({
            "model": model_tag,
            "dataset": dataset,
            "patient_id": patient_id,
            "rank": rerank_position,
            "disease": disease,
            "match_score": match_score,
            "original_rank": original_rank,
            "source_rerank_file": rerank_file
        })

    top10_df = pd.DataFrame(records)

    # Safety: if rerank_position missing, use row order
    if top10_df["rank"].isna().any():
        top10_df["rank"] = range(1, len(top10_df) + 1)

    top10_df["rank"] = pd.to_numeric(top10_df["rank"], errors="coerce")
    top10_df = top10_df.dropna(subset=["rank"]).copy()
    top10_df["rank"] = top10_df["rank"].astype(int)

    top10_df = top10_df.sort_values("rank").head(top_n).copy()

    # Save patient-level top10
    pt_output_folder = os.path.join(
        top10_output_root,
        safe_filename(model_tag),
        safe_filename(dataset),
        safe_filename(patient_id)
    )

    os.makedirs(pt_output_folder, exist_ok=True)

    top10_file = os.path.join(
        pt_output_folder,
        f"{safe_filename(patient_id)}_{safe_filename(dataset)}_{safe_filename(model_tag)}_embedding_top10.csv"
    )

    top10_df.to_csv(top10_file, index=False, encoding="utf-8-sig")

    print(f"✅ Saved top10: {top10_file}")

    top10_detail_dfs.append(top10_df)

    top10_summary_records.append({
        "model": model_tag,
        "dataset": dataset,
        "patient_id": patient_id,
        "status": "saved",
        "n_reranked": len(reranked),
        "top10_file": top10_file,
        "source_file": rerank_file,
        "error": ""
    })


# ============================================================
# Save combined top10 output
# ============================================================

if top10_detail_dfs:
    all_top10_df = pd.concat(top10_detail_dfs, ignore_index=True)
else:
    all_top10_df = pd.DataFrame()

combined_top10_file = os.path.join(
    top10_output_root,
    safe_filename(model_tag),
    f"{safe_filename(model_tag)}_all_patients_embedding_top10.csv"
)

os.makedirs(os.path.dirname(combined_top10_file), exist_ok=True)

all_top10_df.to_csv(
    combined_top10_file,
    index=False,
    encoding="utf-8-sig"
)

top10_summary_df = pd.DataFrame(top10_summary_records)

top10_summary_file = os.path.join(
    top10_output_root,
    safe_filename(model_tag),
    f"{safe_filename(model_tag)}_embedding_top10_save_summary.csv"
)

top10_summary_df.to_csv(
    top10_summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 100)
print(f"✅ Combined top10 saved: {combined_top10_file}")
print(f"✅ Top10 save summary saved: {top10_summary_file}")


✅ Input metadata lookup records: 20
📂 Embedding reranker output root: /Users/sandersu1/downloads/dataset_json_output_claude
📄 Found reranked files: 10
----------------------------------------------------------------------------------------------------
📥 [1/10] Reading:
/Users/sandersu1/downloads/dataset_json_output_claude/6142631-1_pubmed_cosine_reranked.txt
✅ Saved top10: /Users/sandersu1/downloads/embedding_reranker_top10_local/claude/pubmed/6142631-1/6142631-1_pubmed_claude_embedding_top10.csv
----------------------------------------------------------------------------------------------------
📥 [2/10] Reading:
/Users/sandersu1/downloads/dataset_json_output_claude/6219329-1_pubmed_cosine_reranked.txt
✅ Saved top10: /Users/sandersu1/downloads/embedding_reranker_top10_local/claude/pubmed/6219329-1/6219329-1_pubmed_claude_embedding_top10.csv
----------------------------------------------------------------------------------------------------
📥 [3/10] Reading:
/Users/sandersu1/downloads/d

## Prepare NLP ScispaCy Pipeline to parse case reports to get UMLS concepts


In [9]:
%pip install scispacy
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_scibert-0.5.4.tar.gz
%pip install tiktoken

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_scibert-0.5.4.tar.gz (417.0 MB)
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


## Load NLP Pipeline

In [11]:
import spacy
from scispacy.linking import EntityLinker
from transformers import AutoTokenizer
import tiktoken
import warnings
import time
import traceback
from collections import defaultdict

print("🔄 Loading UMLS spaCy pipeline...")

nlp_umls = spacy.load("en_core_sci_scibert")

nlp_umls.add_pipe("scispacy_linker", name="scispacy_umls_linker", config={
    "threshold": 0.8,
    "resolve_abbreviations": True,
    "linker_name": "umls"
})
umls_linker = nlp_umls.get_pipe("scispacy_umls_linker")

if "sentencizer" not in nlp_umls.pipe_names:
    nlp_umls.add_pipe("sentencizer")

hf_tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
enc = tiktoken.get_encoding("cl100k_base")

🔄 Loading UMLS spaCy pipeline...


/opt/anaconda3/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warn

## Load UMLS mapping file

In [13]:
sem_type_df = pd.read_excel(
    "./utilities/umls_mapping.xlsx",
    header=None,
    names=["TUI", "Type_name"]
)
sem_type_map = dict(zip(sem_type_df["TUI"], sem_type_df["Type_name"]))
display(sem_type_df)

,TUI,Type_name
0,T001,Organism
1,T002,Plant
2,T004,Fungus
3,T005,Virus
4,T007,Bacterium
...,...,...
122,T197,Inorganic Chemical
123,T200,Clinical Drug
124,T201,Clinical Attribute
125,T203,Drug Delivery Device


## Main UMLS Parse function

In [15]:
def link_umls_entities(nlp_model, text, model, patient_id,dataset,linker,max_tokens=512):

    try:
        if not isinstance(text, str) or not text.strip():
            raise ValueError("Empty or invalid text content.")

        wordpiece_len = len(hf_tokenizer.tokenize(text))
        token_chunks = []
        
        if wordpiece_len <= max_tokens:
            doc = nlp_model(text)
            token_chunks.append((0, doc, 0))
        else:
            doc_full = nlp_model.make_doc(text)
            doc_full = nlp_model.get_pipe("sentencizer")(doc_full)

            current_text = ""
            current_start = None
            group_idx = 0
            

            for sent in doc_full.sents:
                sent_text = sent.text.strip()
                if not sent_text:
                    continue

                if current_start is None:
                    current_start = sent.start_char

                combined_text = f"{current_text} {sent_text}".strip() if current_text else sent_text
                wp_len = len(hf_tokenizer.tokenize(combined_text))
                
                if wp_len <= max_tokens:
                    current_text = combined_text
                    
                    
                else:
                    if current_text.strip():
                        chunk_doc = nlp_model(current_text)
                        token_chunks.append((group_idx, chunk_doc, current_start))
                        group_idx += 1
                    current_text = sent_text
                    current_start = sent.start_char

            if current_text.strip():
                chunk_doc = nlp_model(current_text)
                token_chunks.append((group_idx, chunk_doc, current_start))
                

        results = []
        for chunk_index, chunk_doc, chunk_offset in token_chunks:
            for entity in chunk_doc.ents:
                try:
                    for cui, score in entity._.kb_ents:
                        concept = linker.kb.cui_to_entity.get(cui)

                        sentence_span = entity.sent
                        sentence_text = sentence_span.text.strip() if sentence_span else ""
                        sentence_start = chunk_offset + sentence_span.start_char if sentence_span else None
                        sentence_end = chunk_offset + sentence_span.end_char if sentence_span else None

                        tui_list = concept.types  # e.g., ["T109", "T121"]
                        type_names = [sem_type_map.get(tui, tui) for tui in tui_list]

                        results.append({
                            "model": model,
                            'patient_id': patient_id,
                            'dataset': dataset,
                            "chunk_index": chunk_index,
                            "UMLS_name": concept.canonical_name,
                            "CUI": cui,
                            "similarity": score,
                            "semantic_type": tui_list,
                            "semantic_type_name": "| ".join(type_names),
                            "matched_text": entity.text,
                            "start_char": chunk_offset + entity.start_char,
                            "end_char": chunk_offset + entity.end_char,
                            "sentence": sentence_text,
                            "sentence_start": sentence_start,
                            "sentence_end": sentence_end
                        })
                except Exception:
                    traceback.print_exc()

        return pd.DataFrame(results)

    except Exception as e:
        print(f"Skipping text due to parsing error: {e}")
        return pd.DataFrame()


## Parse case reports to get UMLS concepts

In [17]:

# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    """
    Make patient_id / dataset safe for folder or file names.
    Example: 5941722-1 stays 5941722-1
    """
    x = str(x)
    x = re.sub(r"[^\w\-\.]+", "_", x)
    return x.strip("_")


def load_json_file(json_path):
    """
    Load one JSON file.
    Expected format:
    {
      "patient_id": "5941722-1",
      "dataset": "pubmed",
      "case_data": "xxxx"
    }
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return data


# ============================================================
# Main processing
# ============================================================

json_files = sorted(glob.glob(os.path.join(input_root, "*.json")))

print(f"📂 Found {len(json_files)} JSON files in: {input_root}")
print(f"📁 Output root: {output_root}")

summary_records = []

for i, json_path in enumerate(json_files, 1):

    print("=" * 100)
    print(f"📥 Processing file {i}/{len(json_files)}: {json_path}")

    try:
        data = load_json_file(json_path)
    except Exception as e:
        print(f"❌ Failed to read JSON: {json_path}")
        print(f"   Error: {e}")
        continue

    # If a file accidentally contains a list of patient records, handle it too
    if isinstance(data, list):
        patient_records = data
    else:
        patient_records = [data]

    for rec_idx, record in enumerate(patient_records, 1):

        patient_id = str(record.get("patient_id", "")).strip()
        dataset = str(record.get("dataset", "")).strip()
        case_data = record.get("case_data", "")

        if case_data is None:
            case_data = ""

        case_data = str(case_data).strip()

        if not patient_id:
            print(f"⚠️ Missing patient_id in file: {json_path}")
            continue

        if not dataset:
            print(f"⚠️ Missing dataset for patient {patient_id}; using 'unknown_dataset'")
            dataset = "unknown_dataset"

        if not case_data:
            print(f"⚠️ Empty case_data for patient {patient_id}; skipping")
            summary_records.append({
                "patient_id": patient_id,
                "dataset": dataset,
                "source_file": os.path.basename(json_path),
                "status": "skipped_empty_case_data",
                "n_umls_records": 0,
                "output_file": ""
            })
            continue

        print(f"🧾 patient_id: {patient_id}")
        print(f"🧾 dataset: {dataset}")
        print(f"🧾 case_data length: {len(case_data)} characters")

        # ------------------------------------------------------------
        # Create patient-specific output folder
        # ------------------------------------------------------------
        

        dataset_folder = os.path.join(output_root, safe_filename(dataset))
        pt_folder = os.path.join(dataset_folder, safe_filename(patient_id))

        os.makedirs(pt_folder, exist_ok=True)

        output_csv = os.path.join(
            pt_folder,
            f"{safe_filename(patient_id)}_{safe_filename(dataset)}_case_data_umls_concepts.csv"
        )

        # ------------------------------------------------------------
        # Parse UMLS concepts from case_data
        # ------------------------------------------------------------
        try:
            df_linked = link_umls_entities(
                nlp_model=nlp_umls,
                text=case_data,
                model=model_tag,
                patient_id=patient_id,
                linker=umls_linker,
                dataset=dataset
            )
        except Exception as e:
            print(f"❌ UMLS linking failed for patient {patient_id}")
            print(f"   Error: {e}")

            summary_records.append({
                "patient_id": patient_id,
                "dataset": dataset,
                "source_file": os.path.basename(json_path),
                "status": "umls_linking_failed",
                "n_umls_records": 0,
                "output_file": ""
            })
            continue

        # ------------------------------------------------------------
        # Save output as file, not Spark table
        # ------------------------------------------------------------
        if df_linked is None or df_linked.empty:
            print(f"⚠️ No UMLS concepts matched for patient {patient_id}")

            # Save an empty CSV so every patient has an output file
            empty_df = pd.DataFrame([{
                "patient_id": patient_id,
                "dataset": dataset,
                "model": model_tag,
                "source_file": os.path.basename(json_path),
                "note": "No UMLS concepts matched"
            }])

            empty_df.to_csv(output_csv, index=False)

            summary_records.append({
                "patient_id": patient_id,
                "dataset": dataset,
                "source_file": os.path.basename(json_path),
                "status": "no_umls_concepts",
                "n_umls_records": 0,
                "output_file": output_csv
            })

            continue

        # Make sure metadata columns exist
        df_linked = df_linked.copy()

        df_linked["patient_id"] = patient_id
        df_linked["dataset"] = dataset
        df_linked["model"] = model_tag
        df_linked["source_file"] = os.path.basename(json_path)

        df_linked.to_csv(output_csv, index=False, encoding="utf-8-sig")

        print(f"✅ Saved {len(df_linked)} UMLS records")
        print(f"   Output: {output_csv}")

        summary_records.append({
            "patient_id": patient_id,
            "dataset": dataset,
            "source_file": os.path.basename(json_path),
            "status": "saved",
            "n_umls_records": len(df_linked),
            "output_file": output_csv
        })


# ============================================================
# Save processing summary
# ============================================================

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(
    output_root,
    f"umls_case_data_processing_summary_{model_tag}.csv"
)

summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("=" * 100)
print("🎉 Completed all patient JSON files.")
print(f"📄 Summary saved to: {summary_path}")

if not summary_df.empty:
    print("\nSummary counts:")
    print(summary_df["status"].value_counts())

📂 Found 10 JSON files in: /Users/sandersu1/downloads/test_pts
📁 Output root: /Users/sandersu1/downloads/dataset_json_output_claude
📥 Processing file 1/10: /Users/sandersu1/downloads/test_pts/6142631-1_pubmed.json
🧾 patient_id: 6142631-1
🧾 dataset: pubmed
🧾 case_data length: 2445 characters


/opt/anaconda3/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/opt/anaconda3/lib/python3.12/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


✅ Saved 291 UMLS records
   Output: /Users/sandersu1/downloads/dataset_json_output_claude/pubmed/6142631-1/6142631-1_pubmed_case_data_umls_concepts.csv
📥 Processing file 2/10: /Users/sandersu1/downloads/test_pts/6219329-1_pubmed.json
🧾 patient_id: 6219329-1
🧾 dataset: pubmed
🧾 case_data length: 2414 characters


/opt/anaconda3/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/opt/anaconda3/lib/python3.12/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


✅ Saved 229 UMLS records
   Output: /Users/sandersu1/downloads/dataset_json_output_claude/pubmed/6219329-1/6219329-1_pubmed_case_data_umls_concepts.csv
📥 Processing file 3/10: /Users/sandersu1/downloads/test_pts/6257492-1_pubmed.json
🧾 patient_id: 6257492-1
🧾 dataset: pubmed
🧾 case_data length: 2106 characters


...

/opt/anaconda3/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/opt/anaconda3/lib/python3.12/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/opt/anaconda3/lib/python3.12/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


🧾 patient_id: 6935327-1
🧾 dataset: pubmed
🧾 case_data length: 2783 characters
✅ Saved 337 UMLS records
   Output: /Users/sandersu1/downloads/dataset_json_output_claude/pubmed/6935327-1/6935327-1_pubmed_case_data_umls_concepts.csv
📥 Processing file 9/10: /Users/sandersu1/downloads/test_pts/7007742-1_pubmed.json
🧾 patient_id: 7007742-1
🧾 dataset: pubmed
🧾 case_data length: 4611 characters


/opt/anaconda3/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/opt/anaconda3/lib/python3.12/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


✅ Saved 598 UMLS records
   Output: /Users/sandersu1/downloads/dataset_json_output_claude/pubmed/7007742-1/7007742-1_pubmed_case_data_umls_concepts.csv
📥 Processing file 10/10: /Users/sandersu1/downloads/test_pts/7058835-1_pubmed.json
🧾 patient_id: 7058835-1
🧾 dataset: pubmed
🧾 case_data length: 1164 characters
✅ Saved 139 UMLS records
   Output: /Users/sandersu1/downloads/dataset_json_output_claude/pubmed/7058835-1/7058835-1_pubmed_case_data_umls_concepts.csv
🎉 Completed all patient JSON files.
📄 Summary saved to: /Users/sandersu1/downloads/dataset_json_output_claude/umls_case_data_processing_summary_claude.csv

Summary counts:
saved    10
Name: status, dtype: int64


/opt/anaconda3/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/opt/anaconda3/lib/python3.12/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


## Only keep interested UMLS concepts

In [19]:

# ============================================================
# Settings
# ============================================================


target_types = [
    "Disease or Syndrome", "Gene or Genome", "Therapeutic or Preventive Procedure",
    "Pathologic Function", "Diagnostic Procedure", "Sign or Symptom", "Neoplastic Process",
    "Organic Chemical", "Laboratory Procedure", "Congenital Abnormality",
    "Mental or Behavioral Dysfunction", "Pharmacologic Substance", "Genetic Function",
    "Amino Acid, Peptide, or Protein", "Anatomical Abnormality", "Laboratory or Test Result",
    "Clinical Drug", "Enzyme", "Cell or Molecular Dysfunction", "Finding"
]

models = ["claude"]
similarity_threshold = 0.8


# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    x = str(x)
    x = re.sub(r"[^\w\-\.]+", "_", x)
    return x.strip("_")


def has_target_semantic_type(semantic_type_value, target_types):
    """
    semantic_type_name may look like:
    'Disease or Syndrome|Finding'
    """
    if pd.isna(semantic_type_value):
        return False

    semtypes = [x.strip() for x in str(semantic_type_value).split("|")]
    return any(t in semtypes for t in target_types)


def is_negated_umls_name(umls_name):
    """
    Exclude concepts starting with:
    not, no, without
    """
    if pd.isna(umls_name):
        return False

    name = str(umls_name).strip().lower()

    return (
        name.startswith("not ") or
        name.startswith("no ") or
        name.startswith("without ")
    )


def extract_dataset_patient_from_path(csv_path, output_root):
    """
    Expected path:
    output_root/dataset/patient_id/file.csv
    """
    rel_path = os.path.relpath(csv_path, output_root)
    parts = rel_path.split(os.sep)

    dataset = parts[0] if len(parts) > 0 else "unknown_dataset"
    patient_id = parts[1] if len(parts) > 1 else "unknown_patient"

    return dataset, patient_id


# ============================================================
# Main filtering
# ============================================================

all_model_summaries = []

for model in models:
    print("=" * 100)
    print(f"🚀 Processing model: {model}")

    output_root = os.path.join(base_dir, f"dataset_json_output_{model}")

    filtered_root = os.path.join(
        base_dir,
        f"dataset_json_output_{model}_filtered"
    )

    os.makedirs(filtered_root, exist_ok=True)

    # Find all UMLS concept files under:
    # dataset_json_output_model/dataset/patient_id/*.csv
    csv_files = sorted(
        glob.glob(
            os.path.join(output_root, "*", "*", "*_case_data_umls_concepts.csv")
        )
    )

    print(f"📂 Input root: {output_root}")
    print(f"📄 Found {len(csv_files)} UMLS concept files")

    model_filtered_dfs = []
    model_summary = []

    for i, csv_path in enumerate(csv_files, 1):
        print(f"\n📥 [{i}/{len(csv_files)}] Reading: {csv_path}")

        dataset, patient_id = extract_dataset_patient_from_path(csv_path, output_root)

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"❌ Failed to read CSV: {csv_path}")
            print(f"   Error: {e}")
            continue

        original_n = len(df)

        if df.empty:
            print(f"⚠️ Empty file: {csv_path}")
            filtered_df = df.copy()
        else:
            # ------------------------------------------------------------
            # Make sure required columns exist
            # ------------------------------------------------------------
            required_cols = ["similarity", "semantic_type_name", "UMLS_name"]

            missing_cols = [c for c in required_cols if c not in df.columns]

            if missing_cols:
                print(f"⚠️ Missing required columns: {missing_cols}")
                print("   Skipping this file.")

                model_summary.append({
                    "model": model,
                    "dataset": dataset,
                    "patient_id": patient_id,
                    "source_file": os.path.basename(csv_path),
                    "status": "skipped_missing_columns",
                    "original_n": original_n,
                    "filtered_n": 0,
                    "output_file": ""
                })

                continue

            # ------------------------------------------------------------
            # Convert similarity to numeric
            # ------------------------------------------------------------
            df["similarity"] = pd.to_numeric(df["similarity"], errors="coerce")

            # ------------------------------------------------------------
            # Filter similarity >= threshold
            # ------------------------------------------------------------
            filtered_df = df[df["similarity"] >= similarity_threshold].copy()

            # ------------------------------------------------------------
            # Filter semantic types
            # ------------------------------------------------------------
            filtered_df = filtered_df[
                filtered_df["semantic_type_name"].apply(
                    lambda x: has_target_semantic_type(x, target_types)
                )
            ].copy()

            # ------------------------------------------------------------
            # Exclude negated concepts
            # ------------------------------------------------------------
            filtered_df = filtered_df[
                ~filtered_df["UMLS_name"].apply(is_negated_umls_name)
            ].copy()

            # ------------------------------------------------------------
            # Add metadata if missing
            # ------------------------------------------------------------
            if "model" not in filtered_df.columns:
                filtered_df["model"] = model

            if "dataset" not in filtered_df.columns:
                filtered_df["dataset"] = dataset

            if "patient_id" not in filtered_df.columns:
                filtered_df["patient_id"] = patient_id

            filtered_df["source_concept_file"] = os.path.basename(csv_path)

        # ------------------------------------------------------------
        # Save patient-level filtered file
        # Keep same structure:
        # dataset_json_output_model_filtered/dataset/patient_id/file.csv
        # ------------------------------------------------------------
        filtered_dataset_folder = os.path.join(filtered_root, safe_filename(dataset))
        filtered_pt_folder = os.path.join(filtered_dataset_folder, safe_filename(patient_id))
        os.makedirs(filtered_pt_folder, exist_ok=True)

        filtered_csv_path = os.path.join(
            filtered_pt_folder,
            f"{safe_filename(patient_id)}_{safe_filename(dataset)}_filtered_umls_concepts.csv"
        )

        filtered_df.to_csv(filtered_csv_path, index=False, encoding="utf-8-sig")

        print(f"✅ Original records: {original_n}")
        print(f"✅ Filtered records: {len(filtered_df)}")
        print(f"💾 Saved: {filtered_csv_path}")

        if not filtered_df.empty:
            model_filtered_dfs.append(filtered_df)

        model_summary.append({
            "model": model,
            "dataset": dataset,
            "patient_id": patient_id,
            "source_file": os.path.basename(csv_path),
            "status": "saved",
            "original_n": original_n,
            "filtered_n": len(filtered_df),
            "output_file": filtered_csv_path
        })

    # ============================================================
    # Save combined filtered file for this model
    # ============================================================

    if model_filtered_dfs:
        combined_df = pd.concat(model_filtered_dfs, ignore_index=True)
    else:
        combined_df = pd.DataFrame()

    combined_path = os.path.join(
        filtered_root,
        f"{model}_filtered_UMLS_results_paper_revision.csv"
    )

    combined_df.to_csv(combined_path, index=False, encoding="utf-8-sig")

    print("\n" + "-" * 100)
    print(f"✅ Finished model: {model}")
    print(f"📄 Combined filtered file saved to:")
    print(combined_path)
    print(f"📊 Combined filtered records: {len(combined_df)}")

    # ============================================================
    # Save summary for this model
    # ============================================================

    summary_df = pd.DataFrame(model_summary)

    summary_path = os.path.join(
        filtered_root,
        f"{model}_filtered_UMLS_processing_summary.csv"
    )

    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print(f"📄 Summary saved to:")
    print(summary_path)

    all_model_summaries.append(summary_df)


# ============================================================
# Save all-model summary
# ============================================================

if all_model_summaries:
    all_summary_df = pd.concat(all_model_summaries, ignore_index=True)
else:
    all_summary_df = pd.DataFrame()

all_summary_path = os.path.join(
    base_dir,
    "all_models_filtered_UMLS_processing_summary.csv"
)

all_summary_df.to_csv(all_summary_path, index=False, encoding="utf-8-sig")

print("=" * 100)
print("🎉 Completed all models.")
print(f"📄 All-model summary saved to:")
print(all_summary_path)

if not all_summary_df.empty:
    print("\nSummary by model:")
    display(
        all_summary_df.groupby("model")[["original_n", "filtered_n"]]
        .sum()
        .reset_index()
    )

🚀 Processing model: claude
📂 Input root: /Users/sandersu1/downloads/dataset_json_output_claude
📄 Found 10 UMLS concept files

📥 [1/10] Reading: /Users/sandersu1/downloads/dataset_json_output_claude/pubmed/6142631-1/6142631-1_pubmed_case_data_umls_concepts.csv
✅ Original records: 291
✅ Filtered records: 120
💾 Saved: /Users/sandersu1/downloads/dataset_json_output_claude_filtered/pubmed/6142631-1/6142631-1_pubmed_filtered_umls_concepts.csv

📥 [2/10] Reading: /Users/sandersu1/downloads/dataset_json_output_claude/pubmed/6219329-1/6219329-1_pubmed_case_data_umls_concepts.csv
✅ Original records: 229
✅ Filtered records: 75
💾 Saved: /Users/sandersu1/downloads/dataset_json_output_claude_filtered/pubmed/6219329-1/6219329-1_pubmed_filtered_umls_concepts.csv

📥 [3/10] Reading: /Users/sandersu1/downloads/dataset_json_output_claude/pubmed/6257492-1/6257492-1_pubmed_case_data_umls_concepts.csv
✅ Original records: 313
✅ Filtered records: 161
💾 Saved: /Users/sandersu1/downloads/dataset_json_output_claud

,model,original_n,filtered_n
0,claude,3041,1259


## Calcuate BM25 sum scores for Patients

In [21]:

# ============================================================
# Settings
# ============================================================

utilities_dir = os.path.join(base_dir, "utilities")

top_n = 10

# Patient UMLS concept folder from previous filtering step
patient_umls_root_template = os.path.join(
    base_dir,
    "dataset_json_output_{model}_filtered"
)

# Model's BM25 file location:
# base_dir/utilities/{model}_BM25.xlsx
bm25_file_template = os.path.join(
    utilities_dir,
    "{model}_BM25.xlsx"
)

# New output folder
top10_output_root = os.path.join(
    base_dir,
    "patient_top10_diseases_bm25_local"
)

os.makedirs(top10_output_root, exist_ok=True)


# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    x = str(x)
    x = re.sub(r"[^\w\-\.]+", "_", x)
    return x.strip("_")


def normalize_colname(c):
    return (
        str(c)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def standardize_bm25_columns(df):
    """
    Expected BM25 file columns may look like:
    disease, semantic_t, cui, UMLS_nam, bm25

    This function standardizes them to:
    disease, semantic_type_name, cui, UMLS_name, bm25
    """

    original_cols = list(df.columns)
    rename_map = {}

    for c in original_cols:
        norm = normalize_colname(c)

        if norm == "disease":
            rename_map[c] = "disease"

        elif norm in ["cui", "umls_cui"]:
            rename_map[c] = "cui"

        elif norm.startswith("umls_nam") or norm in ["umls_name", "umls_names"]:
            rename_map[c] = "UMLS_name"

        elif norm.startswith("semantic_t") or norm in ["semantic_type_name", "semantic_type"]:
            rename_map[c] = "semantic_type_name"

        elif norm == "bm25" or "bm25" in norm:
            rename_map[c] = "bm25"

    df = df.rename(columns=rename_map)

    required_cols = ["disease", "cui", "bm25"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"BM25 file missing required columns: {missing_cols}\n"
            f"Original columns: {original_cols}\n"
            f"After rename: {list(df.columns)}"
        )

    df["disease"] = df["disease"].astype(str).str.strip()
    df["cui"] = df["cui"].astype(str).str.strip()
    df["bm25"] = pd.to_numeric(df["bm25"], errors="coerce").fillna(0)

    df = df[
        (df["disease"] != "") &
        (df["disease"].str.lower() != "nan") &
        (df["cui"] != "") &
        (df["cui"].str.lower() != "nan")
    ].copy()

    return df


def standardize_patient_umls_columns(df):
    """
    Patient UMLS file must contain CUI column.
    It may be named:
    cui, CUI, umls_cui, etc.
    """

    original_cols = list(df.columns)
    rename_map = {}

    for c in original_cols:
        norm = normalize_colname(c)

        if norm in ["cui", "umls_cui"]:
            rename_map[c] = "cui"

        elif norm.startswith("umls_nam") or norm in ["umls_name", "umls_names"]:
            rename_map[c] = "patient_UMLS_name"

        elif norm.startswith("semantic_t") or norm in ["semantic_type_name", "semantic_type"]:
            rename_map[c] = "patient_semantic_type_name"

        elif norm == "similarity":
            rename_map[c] = "similarity"

        elif norm == "patient_id":
            rename_map[c] = "patient_id"

        elif norm == "dataset":
            rename_map[c] = "dataset"

    df = df.rename(columns=rename_map)

    if "cui" not in df.columns:
        raise ValueError(
            f"Patient UMLS file missing CUI column.\n"
            f"Original columns: {original_cols}\n"
            f"After rename: {list(df.columns)}"
        )

    df["cui"] = df["cui"].astype(str).str.strip()

    df = df[
        (df["cui"] != "") &
        (df["cui"].str.lower() != "nan")
    ].copy()

    return df


def extract_dataset_patient_from_path(csv_path, patient_umls_root):
    """
    Expected path:
    dataset_json_output_model_filtered/dataset/patient_id/file.csv
    """

    rel_path = os.path.relpath(csv_path, patient_umls_root)
    parts = rel_path.split(os.sep)

    dataset = parts[0] if len(parts) >= 1 else "unknown_dataset"
    patient_id = parts[1] if len(parts) >= 2 else "unknown_patient"

    return dataset, patient_id


# ============================================================
# Main processing
# ============================================================

all_summary_records = []

for model in models:

    print("=" * 100)
    print(f"🚀 Processing model: {model}")

    bm25_path = bm25_file_template.format(model=model)
    patient_umls_root = patient_umls_root_template.format(model=model)

    model_output_root = os.path.join(
        top10_output_root,
        safe_filename(model)
    )

    os.makedirs(model_output_root, exist_ok=True)

    # ------------------------------------------------------------
    # Check files/folders
    # ------------------------------------------------------------
    if not os.path.exists(bm25_path):
        print(f"⚠️ BM25 file not found, skipping model:")
        print(bm25_path)
        continue

    if not os.path.exists(patient_umls_root):
        print(f"⚠️ Patient UMLS folder not found, skipping model:")
        print(patient_umls_root)
        continue

    # ------------------------------------------------------------
    # Read BM25 Excel file
    # ------------------------------------------------------------
    print(f"📥 Reading BM25 file:")
    print(bm25_path)

    bm25_df = pd.read_excel(bm25_path)
    bm25_df = standardize_bm25_columns(bm25_df)

    # Optional: keep only positive BM25
    bm25_df = bm25_df[bm25_df["bm25"] > 0].copy()

    print(f"✅ BM25 records: {len(bm25_df)}")
    print(f"✅ BM25 diseases: {bm25_df['disease'].nunique()}")
    print(f"✅ BM25 CUIs: {bm25_df['cui'].nunique()}")

    # ------------------------------------------------------------
    # Automatically scan all patient UMLS files
    # No dataset_list needed
    # ------------------------------------------------------------
    csv_files = sorted(
        glob.glob(
            os.path.join(
                patient_umls_root,
                "*",
                "*",
                "*_filtered_umls_concepts.csv"
            )
        )
    )

    print(f"📂 Patient UMLS root:")
    print(patient_umls_root)
    print(f"📄 Found {len(csv_files)} patient UMLS files")

    model_all_top10 = []
    model_summary_records = []

    for i, csv_path in enumerate(csv_files, 1):

        dataset, patient_id = extract_dataset_patient_from_path(
            csv_path,
            patient_umls_root
        )

        print("-" * 100)
        print(f"📥 [{i}/{len(csv_files)}] model={model}, dataset={dataset}, patient_id={patient_id}")
        print(csv_path)

        try:
            pt_df = pd.read_csv(csv_path)
            pt_df = standardize_patient_umls_columns(pt_df)
        except Exception as e:
            print(f"❌ Failed to read/process patient UMLS file")
            print(f"   Error: {e}")

            model_summary_records.append({
                "model": model,
                "dataset": dataset,
                "patient_id": patient_id,
                "status": "failed_read_patient_umls",
                "n_patient_cuis": 0,
                "n_joined_records": 0,
                "n_top10": 0,
                "output_file": ""
            })

            continue

        # ------------------------------------------------------------
        # Unique patient CUIs
        # ------------------------------------------------------------
        patient_cuis = (
            pt_df[["cui"]]
            .dropna()
            .drop_duplicates()
            .copy()
        )

        n_patient_cuis = len(patient_cuis)

        if n_patient_cuis == 0:
            print(f"⚠️ No patient CUIs found")

            model_summary_records.append({
                "model": model,
                "dataset": dataset,
                "patient_id": patient_id,
                "status": "no_patient_cuis",
                "n_patient_cuis": 0,
                "n_joined_records": 0,
                "n_top10": 0,
                "output_file": ""
            })

            continue

        # ------------------------------------------------------------
        # Join patient CUIs with BM25 disease-CUI file
        # Equivalent to Spark join on CUI
        # ------------------------------------------------------------
        joined = patient_cuis.merge(
            bm25_df,
            on="cui",
            how="inner"
        )

        n_joined_records = len(joined)

        if joined.empty:
            print(f"⚠️ No BM25 disease matches")

            model_summary_records.append({
                "model": model,
                "dataset": dataset,
                "patient_id": patient_id,
                "status": "no_bm25_matches",
                "n_patient_cuis": n_patient_cuis,
                "n_joined_records": 0,
                "n_top10": 0,
                "output_file": ""
            })

            continue

        # ------------------------------------------------------------
        # Aggregate by matched disease
        # shared_cui_count = countDistinct(cui)
        # bm25_sum = sum(bm25)
        # ------------------------------------------------------------
        agg_df = (
            joined
            .groupby("disease", as_index=False)
            .agg(
                shared_cui_count=("cui", "nunique"),
                bm25_sum=("bm25", "sum"),
                matched_cuis=("cui", lambda x: " | ".join(sorted(set(map(str, x)))))
            )
        )

        if "UMLS_name" in joined.columns:
            matched_names = (
                joined
                .groupby("disease")["UMLS_name"]
                .apply(lambda x: " | ".join(sorted(set(map(str, x)))))
                .reset_index()
                .rename(columns={"UMLS_name": "matched_umls_names"})
            )

            agg_df = agg_df.merge(matched_names, on="disease", how="left")

        agg_df["model"] = model
        agg_df["dataset"] = dataset
        agg_df["patient_id"] = patient_id

        # ------------------------------------------------------------
        # Rank top 10 diseases by BM25 sum
        # Tie-breaker: shared CUI count, disease name
        # ------------------------------------------------------------
        agg_df = agg_df.sort_values(
            by=["bm25_sum", "shared_cui_count", "disease"],
            ascending=[False, False, True]
        ).reset_index(drop=True)

        agg_df["rank"] = range(1, len(agg_df) + 1)

        top10_df = agg_df[agg_df["rank"] <= top_n].copy()

        # Reorder columns
        first_cols = [
            "model",
            "dataset",
            "patient_id",
            "rank",
            "disease",
            "shared_cui_count",
            "bm25_sum",
            "matched_cuis"
        ]

        other_cols = [c for c in top10_df.columns if c not in first_cols]
        top10_df = top10_df[first_cols + other_cols]

        # ------------------------------------------------------------
        # Save patient-level top10 result
        # ------------------------------------------------------------
        pt_output_folder = os.path.join(
            model_output_root,
            safe_filename(dataset),
            safe_filename(patient_id)
        )

        os.makedirs(pt_output_folder, exist_ok=True)

        pt_output_file = os.path.join(
            pt_output_folder,
            f"{safe_filename(patient_id)}_{safe_filename(dataset)}_{safe_filename(model)}_top10_diseases_bm25.csv"
        )

        top10_df.to_csv(
            pt_output_file,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"✅ Patient CUIs: {n_patient_cuis}")
        print(f"✅ Joined BM25 records: {n_joined_records}")
        print(f"✅ Top10 diseases: {len(top10_df)}")
        print(f"💾 Saved:")
        print(pt_output_file)

        model_all_top10.append(top10_df)

        model_summary_records.append({
            "model": model,
            "dataset": dataset,
            "patient_id": patient_id,
            "status": "saved",
            "n_patient_cuis": n_patient_cuis,
            "n_joined_records": n_joined_records,
            "n_top10": len(top10_df),
            "output_file": pt_output_file
        })

    # ============================================================
    # Save model-level combined output
    # ============================================================

    if model_all_top10:
        model_combined_df = pd.concat(model_all_top10, ignore_index=True)
    else:
        model_combined_df = pd.DataFrame()

    model_combined_file = os.path.join(
        model_output_root,
        f"{safe_filename(model)}_all_patients_top10_diseases_bm25.csv"
    )

    model_combined_df.to_csv(
        model_combined_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("=" * 100)
    print(f"✅ Finished model: {model}")
    print(f"📄 Model-level combined output saved:")
    print(model_combined_file)
    print(f"📊 Rows: {len(model_combined_df)}")

    # ============================================================
    # Save model summary
    # ============================================================

    model_summary_df = pd.DataFrame(model_summary_records)

    model_summary_file = os.path.join(
        model_output_root,
        f"{safe_filename(model)}_bm25_top10_processing_summary.csv"
    )

    model_summary_df.to_csv(
        model_summary_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"📄 Model summary saved:")
    print(model_summary_file)

    all_summary_records.extend(model_summary_records)


# ============================================================
# Save all-model summary
# ============================================================

all_summary_df = pd.DataFrame(all_summary_records)

all_summary_file = os.path.join(
    top10_output_root,
    "all_models_bm25_top10_processing_summary.csv"
)

all_summary_df.to_csv(
    all_summary_file,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 100)
print("🏁 All done!")
print(f"📄 All-model summary saved:")
print(all_summary_file)

if not all_summary_df.empty:
    display(
        all_summary_df.groupby(["model", "dataset"])[
            ["n_patient_cuis", "n_joined_records", "n_top10"]
        ]
        .sum()
        .reset_index()
    )

🚀 Processing model: claude
📥 Reading BM25 file:
/Users/sandersu1/downloads/utilities/claude_BM25.xlsx


/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


✅ BM25 records: 529686
✅ BM25 diseases: 1320
✅ BM25 CUIs: 40575
📂 Patient UMLS root:
/Users/sandersu1/downloads/dataset_json_output_claude_filtered
📄 Found 10 patient UMLS files
----------------------------------------------------------------------------------------------------
📥 [1/10] model=claude, dataset=pubmed, patient_id=6142631-1
/Users/sandersu1/downloads/dataset_json_output_claude_filtered/pubmed/6142631-1/6142631-1_pubmed_filtered_umls_concepts.csv
✅ Patient CUIs: 99
✅ Joined BM25 records: 9848
✅ Top10 diseases: 10
💾 Saved:
/Users/sandersu1/downloads/patient_top10_diseases_bm25_local/claude/pubmed/6142631-1/6142631-1_pubmed_claude_top10_diseases_bm25.csv
----------------------------------------------------------------------------------------------------
📥 [2/10] model=claude, dataset=pubmed, patient_id=6219329-1
/Users/sandersu1/downloads/dataset_json_output_claude_filtered/pubmed/6219329-1/6219329-1_pubmed_filtered_umls_concepts.csv
✅ Patient CUIs: 59
✅ Joined BM25 records: 

,model,dataset,n_patient_cuis,n_joined_records,n_top10
0,claude,pubmed,992,176238,100


## RRF calculation for embedding and BM25 results

In [23]:
# ============================================================
# Settings
# ============================================================

k_rrf = 60
top_n = 10

bm25_root = os.path.join(
    base_dir,
    "patient_top10_diseases_bm25_local"
)

embedding_root = os.path.join(
    base_dir,
    "embedding_reranker_top10_local"
)

truth_file = os.path.join(
    base_dir,
    "utilities",
    "test_pt_correct_disease.xlsx"
)

fusion_output_root = os.path.join(
    base_dir,
    "fusion_dense_sparse_rrf_local"
)

os.makedirs(fusion_output_root, exist_ok=True)


# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    x = str(x)
    x = re.sub(r"[^\w\-\.]+", "_", x)
    return x.strip("_")


def normalize_text(x):
    if pd.isna(x):
        return ""

    x = str(x).strip().lower()
    x = x.replace("’", "'").replace("`", "'")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x


def standardize_bm25_columns(df):
    """
    BM25 top10 expected columns:
    model, dataset, patient_id, rank, disease, bm25_sum, shared_cui_count
    """

    rename_map = {}

    for c in df.columns:
        c_norm = str(c).strip().lower()

        if c_norm == "rank":
            rename_map[c] = "sparse_rank"

        elif c_norm in ["disease", "predicted_disease", "matched_disease"]:
            rename_map[c] = "disease"

        elif c_norm == "patient_id":
            rename_map[c] = "patient_id"

        elif c_norm == "dataset":
            rename_map[c] = "dataset"

        elif c_norm == "model":
            rename_map[c] = "model"

        elif c_norm == "bm25_sum":
            rename_map[c] = "bm25_sum"

        elif c_norm == "shared_cui_count":
            rename_map[c] = "shared_cui_count"

        elif c_norm == "matched_cuis":
            rename_map[c] = "matched_cuis"

        elif c_norm == "matched_umls_names":
            rename_map[c] = "matched_umls_names"

    df = df.rename(columns=rename_map)

    required_cols = ["dataset", "patient_id", "disease", "sparse_rank"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"BM25 file missing columns: {missing_cols}\n"
            f"Current columns: {list(df.columns)}"
        )

    df["dataset"] = df["dataset"].astype(str).str.strip()
    df["patient_id"] = df["patient_id"].astype(str).str.strip()
    df["disease"] = df["disease"].astype(str).str.strip()
    df["sparse_rank"] = pd.to_numeric(df["sparse_rank"], errors="coerce")

    df = df.dropna(subset=["sparse_rank"]).copy()
    df["sparse_rank"] = df["sparse_rank"].astype(int)

    df["disease_norm"] = df["disease"].apply(normalize_text)

    return df


def standardize_embedding_columns(df):
    """
    Embedding top10 expected columns may be either:
    rank, disease
    or:
    rerank_position, predicted_disease
    """

    rename_map = {}

    for c in df.columns:
        c_norm = str(c).strip().lower()

        if c_norm in ["rank", "rerank_position", "dense_rank"]:
            rename_map[c] = "dense_rank"

        elif c_norm in ["disease", "predicted_disease", "matched_disease"]:
            rename_map[c] = "disease"

        elif c_norm == "patient_id":
            rename_map[c] = "patient_id"

        elif c_norm == "dataset":
            rename_map[c] = "dataset"

        elif c_norm == "model":
            rename_map[c] = "model"

        elif c_norm == "match_score":
            rename_map[c] = "dense_match_score"

        elif c_norm == "original_rank":
            rename_map[c] = "dense_original_rank"

    df = df.rename(columns=rename_map)

    required_cols = ["dataset", "patient_id", "disease", "dense_rank"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"Embedding file missing columns: {missing_cols}\n"
            f"Current columns: {list(df.columns)}"
        )

    df["dataset"] = df["dataset"].astype(str).str.strip()
    df["patient_id"] = df["patient_id"].astype(str).str.strip()
    df["disease"] = df["disease"].astype(str).str.strip()
    df["dense_rank"] = pd.to_numeric(df["dense_rank"], errors="coerce")

    df = df.dropna(subset=["dense_rank"]).copy()
    df["dense_rank"] = df["dense_rank"].astype(int)

    df["disease_norm"] = df["disease"].apply(normalize_text)

    return df


def standardize_truth_columns(df):
    rename_map = {}

    for c in df.columns:
        c_norm = str(c).strip().lower()

        if c_norm == "dataset":
            rename_map[c] = "dataset"

        elif c_norm in ["patient_id", "person_id", "pt_id"]:
            rename_map[c] = "patient_id"

        elif c_norm in ["rare_disease_name", "disease", "true_disease", "correct_disease"]:
            rename_map[c] = "rare_disease_name"

    df = df.rename(columns=rename_map)

    required_cols = ["dataset", "patient_id", "rare_disease_name"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"Truth file missing columns: {missing_cols}\n"
            f"Current columns: {list(df.columns)}"
        )

    df = df[["dataset", "patient_id", "rare_disease_name"]].copy()

    df["dataset"] = df["dataset"].astype(str).str.strip()
    df["patient_id"] = df["patient_id"].astype(str).str.strip()
    df["rare_disease_name"] = df["rare_disease_name"].astype(str).str.strip()
    df["true_disease_norm"] = df["rare_disease_name"].apply(normalize_text)

    df = df.drop_duplicates().reset_index(drop=True)

    return df


def load_all_bm25_files(model):
    pattern = os.path.join(
        bm25_root,
        safe_filename(model),
        "*",
        "*",
        "*_top10_diseases_bm25.csv"
    )

    files = sorted(glob.glob(pattern))
    dfs = []

    print(f"📄 BM25 files found for {model}: {len(files)}")

    for path in files:
        try:
            tmp = pd.read_csv(path)
            tmp = standardize_bm25_columns(tmp)
            tmp["source_bm25_file"] = path
            dfs.append(tmp)
        except Exception as e:
            print(f"❌ Failed BM25 file: {path}")
            print(f"   Error: {e}")

    if dfs:
        return pd.concat(dfs, ignore_index=True)

    return pd.DataFrame()


def load_all_embedding_files(model):
    pattern = os.path.join(
        embedding_root,
        safe_filename(model),
        "*",
        "*",
        "*_embedding_top10.csv"
    )

    files = sorted(glob.glob(pattern))
    dfs = []

    print(f"📄 Embedding files found for {model}: {len(files)}")

    for path in files:
        try:
            tmp = pd.read_csv(path)
            tmp = standardize_embedding_columns(tmp)
            tmp["source_embedding_file"] = path
            dfs.append(tmp)
        except Exception as e:
            print(f"❌ Failed embedding file: {path}")
            print(f"   Error: {e}")

    if dfs:
        return pd.concat(dfs, ignore_index=True)

    return pd.DataFrame()


# ============================================================
# Load truth file
# ============================================================

truth_df = pd.read_excel(truth_file)
truth_df = standardize_truth_columns(truth_df)

print(f"✅ Truth records: {len(truth_df)}")
print(f"✅ Truth patients: {truth_df[['dataset', 'patient_id']].drop_duplicates().shape[0]}")

display(truth_df.head())


# ============================================================
# Main RRF fusion
# ============================================================

all_fusion_dfs = []
all_eval_dfs = []
all_summary_records = []

for model in models:

    print("=" * 100)
    print(f"🚀 RRF fusion for model: {model}")

    sparse_df = load_all_bm25_files(model)
    dense_df = load_all_embedding_files(model)

    if sparse_df.empty and dense_df.empty:
        print(f"⚠️ No BM25 or embedding files found for model: {model}")
        continue

    # ------------------------------------------------------------
    # Keep only columns needed from sparse
    # ------------------------------------------------------------
    if not sparse_df.empty:
        sparse_keep_cols = [
            "model", "dataset", "patient_id", "disease_norm",
            "disease", "sparse_rank"
        ]

        for extra_col in [
            "bm25_sum", "shared_cui_count",
            "matched_cuis", "matched_umls_names",
            "source_bm25_file"
        ]:
            if extra_col in sparse_df.columns:
                sparse_keep_cols.append(extra_col)

        sparse_df = sparse_df[sparse_keep_cols].copy()
        sparse_df = sparse_df.rename(columns={"disease": "sparse_disease"})

    # ------------------------------------------------------------
    # Keep only columns needed from dense
    # ------------------------------------------------------------
    if not dense_df.empty:
        dense_keep_cols = [
            "model", "dataset", "patient_id", "disease_norm",
            "disease", "dense_rank"
        ]

        for extra_col in [
            "dense_match_score", "dense_original_rank",
            "source_embedding_file"
        ]:
            if extra_col in dense_df.columns:
                dense_keep_cols.append(extra_col)

        dense_df = dense_df[dense_keep_cols].copy()
        dense_df = dense_df.rename(columns={"disease": "dense_disease"})

    # ------------------------------------------------------------
    # Outer join dense and sparse by patient + disease_norm
    # ------------------------------------------------------------
    if sparse_df.empty:
        joined_df = dense_df.copy()
    elif dense_df.empty:
        joined_df = sparse_df.copy()
    else:
        joined_df = sparse_df.merge(
            dense_df,
            on=["model", "dataset", "patient_id", "disease_norm"],
            how="outer",
            suffixes=("_sparse", "_dense")
        )

    # If model column missing from one side edge cases
    joined_df["model"] = model

    # ------------------------------------------------------------
    # Pick display disease name
    # Prefer dense disease, then sparse disease
    # ------------------------------------------------------------
    if "dense_disease" not in joined_df.columns:
        joined_df["dense_disease"] = np.nan

    if "sparse_disease" not in joined_df.columns:
        joined_df["sparse_disease"] = np.nan

    joined_df["final_disease"] = joined_df["dense_disease"].fillna(
        joined_df["sparse_disease"]
    )

    # ------------------------------------------------------------
    # RRF score
    # RRF = 1 / (rank_dense + k) + 1 / (rank_sparse + k)
    # ------------------------------------------------------------
    joined_df["dense_rrf"] = np.where(
        joined_df["dense_rank"].notna(),
        1.0 / (joined_df["dense_rank"] + k_rrf),
        0.0
    )

    joined_df["sparse_rrf"] = np.where(
        joined_df["sparse_rank"].notna(),
        1.0 / (joined_df["sparse_rank"] + k_rrf),
        0.0
    )

    joined_df["rrf_score"] = joined_df["dense_rrf"] + joined_df["sparse_rrf"]

    joined_df["has_dense"] = joined_df["dense_rank"].notna().astype(int)
    joined_df["has_sparse"] = joined_df["sparse_rank"].notna().astype(int)

    # ------------------------------------------------------------
    # Final ranking per patient
    # Same tie-breaker logic as your Spark version:
    # 1. rrf_score desc
    # 2. prefer dense-supported
    # 3. dense_rank asc
    # 4. sparse_rank asc
    # ------------------------------------------------------------
    joined_df = joined_df.sort_values(
        by=[
            "dataset",
            "patient_id",
            "rrf_score",
            "has_dense",
            "dense_rank",
            "sparse_rank"
        ],
        ascending=[
            True,
            True,
            False,
            False,
            True,
            True
        ],
        na_position="last"
    ).copy()

    joined_df["final_rank"] = (
        joined_df
        .groupby(["dataset", "patient_id"])
        .cumcount() + 1
    )

    final_top10_df = joined_df[joined_df["final_rank"] <= top_n].copy()

    # ------------------------------------------------------------
    # Join truth
    # ------------------------------------------------------------
    final_top10_df = final_top10_df.merge(
        truth_df[["dataset", "patient_id", "rare_disease_name", "true_disease_norm"]],
        on=["dataset", "patient_id"],
        how="left"
    )

    

    # ------------------------------------------------------------
    # Reorder columns
    # ------------------------------------------------------------
    first_cols = [
        "model",
        "dataset",
        "patient_id",
        "rare_disease_name",
        "final_rank",
        "final_disease",
        "disease_norm",
        "rrf_score",
        "dense_rank",
        "sparse_rank",
        "dense_rrf",
        "sparse_rrf",
        "has_dense",
        "has_sparse"
    ]

    other_cols = [c for c in final_top10_df.columns if c not in first_cols]
    final_top10_df = final_top10_df[first_cols + other_cols]

    # ------------------------------------------------------------
    # Save patient-level final top10 CSVs
    # ------------------------------------------------------------
    model_output_root = os.path.join(
        fusion_output_root,
        safe_filename(model)
    )

    os.makedirs(model_output_root, exist_ok=True)

    for (dataset, patient_id), pt_df in final_top10_df.groupby(["dataset", "patient_id"]):

        pt_output_folder = os.path.join(
            model_output_root,
            safe_filename(dataset),
            safe_filename(patient_id)
        )

        os.makedirs(pt_output_folder, exist_ok=True)

        pt_output_file = os.path.join(
            pt_output_folder,
            f"{safe_filename(patient_id)}_{safe_filename(dataset)}_{safe_filename(model)}_rrf_final_top10.csv"
        )

        pt_df.to_csv(
            pt_output_file,
            index=False,
            encoding="utf-8-sig"
        )

    # ------------------------------------------------------------
    # Save model-level combined final top10
    # ------------------------------------------------------------
    model_combined_file = os.path.join(
        model_output_root,
        f"{safe_filename(model)}_all_patients_rrf_final_top10.csv"
    )

    final_top10_df.to_csv(
        model_combined_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"✅ Saved model-level RRF final top10:")
    print(model_combined_file)
    print(f"📊 Rows: {len(final_top10_df)}")

    all_fusion_dfs.append(final_top10_df)

    # ============================================================
    # Optional evaluation: Top1 / Top3 / Top5 for final RRF
    # ============================================================

    eval_records = []

    for (dataset, patient_id), pt_df in final_top10_df.groupby(["dataset", "patient_id"]):

        pt_df = pt_df.sort_values("final_rank").copy()

        truth_sub = truth_df[
            (truth_df["dataset"] == dataset) &
            (truth_df["patient_id"] == patient_id)
        ].copy()

       

        top10_diseases = " | ".join(
            pt_df["final_disease"].astype(str).tolist()
        )

        if truth_sub.empty:
            eval_records.append({
                "model": model,
                "dataset": dataset,
                "patient_id": patient_id,
                "rare_disease_name": None,
                "matched_rank": None,
                "matched_predicted_disease": None,
                "hit_top1": None,
                "hit_top3": None,
                "hit_top5": None,
                "n_predictions": len(pt_df),
                "top10_predicted_diseases": top10_diseases,
                "status": "no_truth_record"
            })
            continue

        true_disease_names = truth_sub["rare_disease_name"].astype(str).tolist()
        true_norm_set = set(truth_sub["true_disease_norm"].tolist())

        pt_df["final_disease_norm_eval"] = pt_df["final_disease"].apply(normalize_text)

        matched = pt_df[
            pt_df["final_disease_norm_eval"].isin(true_norm_set)
        ].copy()

        if matched.empty:
            matched_rank = None
            matched_predicted_disease = None
            hit_top1 = 0
            hit_top3 = 0
            hit_top5 = 0
            status = "not_found_in_top10"

        else:
            best_match = matched.sort_values("final_rank").iloc[0]

            matched_rank = int(best_match["final_rank"])
            matched_predicted_disease = best_match["final_disease"]

            hit_top1 = int(matched_rank <= 1)
            hit_top3 = int(matched_rank <= 3)
            hit_top5 = int(matched_rank <= 5)
            status = "matched"

        eval_records.append({
            "model": model,
            "dataset": dataset,
            "patient_id": patient_id,
            "rare_disease_name": " | ".join(true_disease_names),
            "matched_rank": matched_rank,
            "matched_predicted_disease": matched_predicted_disease,
            "hit_top1": hit_top1,
            "hit_top3": hit_top3,
            "hit_top5": hit_top5,
            "n_predictions": len(pt_df),
            "top10_predicted_diseases": top10_diseases,
            "status": status
        })

    eval_df = pd.DataFrame(eval_records)

    evaluable_df = eval_df[eval_df["status"] != "no_truth_record"].copy()

    if len(evaluable_df) > 0:
        summary_record = {
            "model": model,
            "n_output_patients": len(eval_df),
            "n_evaluable_with_truth": len(evaluable_df),
            "n_no_truth_record": int((eval_df["status"] == "no_truth_record").sum()),
            "top1_hits": int(evaluable_df["hit_top1"].sum()),
            "top3_hits": int(evaluable_df["hit_top3"].sum()),
            "top5_hits": int(evaluable_df["hit_top5"].sum()),
            "recall_at_1": evaluable_df["hit_top1"].mean(),
            "recall_at_3": evaluable_df["hit_top3"].mean(),
            "recall_at_5": evaluable_df["hit_top5"].mean()
        }
    else:
        summary_record = {
            "model": model,
            "n_output_patients": len(eval_df),
            "n_evaluable_with_truth": 0,
            "n_no_truth_record": int((eval_df["status"] == "no_truth_record").sum()) if not eval_df.empty else 0,
            "top1_hits": 0,
            "top3_hits": 0,
            "top5_hits": 0,
            "recall_at_1": None,
            "recall_at_3": None,
            "recall_at_5": None
        }

    summary_df = pd.DataFrame([summary_record])

    eval_file = os.path.join(
        model_output_root,
        f"{safe_filename(model)}_rrf_top1_top3_top5_detail.csv"
    )

    summary_file = os.path.join(
        model_output_root,
        f"{safe_filename(model)}_rrf_top1_top3_top5_summary.csv"
    )

    eval_df.to_csv(eval_file, index=False, encoding="utf-8-sig")
    summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")

    print(f"📄 Eval detail saved:")
    print(eval_file)
    print(f"📄 Eval summary saved:")
    print(summary_file)

    display(summary_df)

    all_eval_dfs.append(eval_df)
    all_summary_records.append(summary_record)


# ============================================================
# Save all-model combined outputs
# ============================================================

if all_fusion_dfs:
    all_fusion_df = pd.concat(all_fusion_dfs, ignore_index=True)
else:
    all_fusion_df = pd.DataFrame()

if all_eval_dfs:
    all_eval_df = pd.concat(all_eval_dfs, ignore_index=True)
else:
    all_eval_df = pd.DataFrame()

all_summary_df = pd.DataFrame(all_summary_records)

all_fusion_file = os.path.join(
    fusion_output_root,
    "all_models_rrf_final_top10.csv"
)

all_eval_file = os.path.join(
    fusion_output_root,
    "all_models_rrf_top1_top3_top5_detail.csv"
)

all_summary_file = os.path.join(
    fusion_output_root,
    "all_models_rrf_top1_top3_top5_summary.csv"
)

all_fusion_df.to_csv(all_fusion_file, index=False, encoding="utf-8-sig")
all_eval_df.to_csv(all_eval_file, index=False, encoding="utf-8-sig")
all_summary_df.to_csv(all_summary_file, index=False, encoding="utf-8-sig")

print("=" * 100)
print("🏁 RRF fusion complete.")
print(f"📄 Final top10 saved: {all_fusion_file}")
print(f"📄 Eval detail saved: {all_eval_file}")
print(f"📄 Eval summary saved: {all_summary_file}")

if not all_summary_df.empty:
    display(all_summary_df)

✅ Truth records: 9290
✅ Truth patients: 9290


,dataset,patient_id,rare_disease_name,true_disease_norm
0,hms,1,Fabry Disease,fabry disease
1,hms,2,Fabry Disease,fabry disease
2,hms,3,Behçets Syndrome,beh ets syndrome
3,hms,4,Behçets Syndrome,beh ets syndrome
4,hms,5,Behçets Syndrome,beh ets syndrome


🚀 RRF fusion for model: claude
📄 BM25 files found for claude: 10
📄 Embedding files found for claude: 10
✅ Saved model-level RRF final top10:
/Users/sandersu1/downloads/fusion_dense_sparse_rrf_local/claude/claude_all_patients_rrf_final_top10.csv
📊 Rows: 100
📄 Eval detail saved:
/Users/sandersu1/downloads/fusion_dense_sparse_rrf_local/claude/claude_rrf_top1_top3_top5_detail.csv
📄 Eval summary saved:
/Users/sandersu1/downloads/fusion_dense_sparse_rrf_local/claude/claude_rrf_top1_top3_top5_summary.csv


,model,n_output_patients,n_evaluable_with_truth,n_no_truth_record,top1_hits,top3_hits,top5_hits,recall_at_1,recall_at_3,recall_at_5
0,claude,10,10,0,7,8,10,0.7,0.8,1.0


🏁 RRF fusion complete.
📄 Final top10 saved: /Users/sandersu1/downloads/fusion_dense_sparse_rrf_local/all_models_rrf_final_top10.csv
📄 Eval detail saved: /Users/sandersu1/downloads/fusion_dense_sparse_rrf_local/all_models_rrf_top1_top3_top5_detail.csv
📄 Eval summary saved: /Users/sandersu1/downloads/fusion_dense_sparse_rrf_local/all_models_rrf_top1_top3_top5_summary.csv


,model,n_output_patients,n_evaluable_with_truth,n_no_truth_record,top1_hits,top3_hits,top5_hits,recall_at_1,recall_at_3,recall_at_5
0,claude,10,10,0,7,8,10,0.7,0.8,1.0


## Generate JSON files based on RRF results for Qwen3-8B reranker reranking

In [25]:

# ============================================================
# Settings
# ============================================================


rrf_root = os.path.join(
    base_dir,
    "fusion_dense_sparse_rrf_local",
    model_tag
)

# Section knowledge file
knowledge_file = os.path.join(
    base_dir,
    "utilities",
    "claude_sections.xlsx"
)

# Patient case_data file
case_data_file = os.path.join(
    base_dir,
    "combined_test_pts.xlsx"
)

json_output_root = os.path.join(
    base_dir,
    "rrf_top10_patient_jsons",
    model_tag
)

os.makedirs(json_output_root, exist_ok=True)

top_n = 10


# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    x = str(x)
    x = re.sub(r"[^\w\-\.]+", "_", x)
    return x.strip("_")


def normalize_text(x):
    if pd.isna(x):
        return ""

    x = str(x).strip().lower()
    x = x.replace("’", "'").replace("`", "'")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x


def normalize_colname(c):
    return (
        str(c)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def normalize_id(x):
    if pd.isna(x):
        return ""

    if isinstance(x, float) and x.is_integer():
        return str(int(x))

    x = str(x).strip()

    if re.fullmatch(r"\d+\.0", x):
        x = x[:-2]

    return x


def normalize_dataset(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def first_nonempty_value(row, candidate_cols):
    for c in candidate_cols:
        if c in row.index:
            val = row[c]
            if not pd.isna(val) and str(val).strip() != "":
                return str(val)
    return ""


# ============================================================
# Standardize RRF columns
# ============================================================

def standardize_rrf_columns(df):
    rename_map = {}

    for c in df.columns:
        c_norm = normalize_colname(c)

        if c_norm == "model":
            rename_map[c] = "model"

        elif c_norm == "dataset":
            rename_map[c] = "dataset"

        elif c_norm in ["patient_id", "person_id", "pt_id"]:
            rename_map[c] = "patient_id"

        elif c_norm in ["final_rank", "rank"]:
            rename_map[c] = "final_rank"

        elif c_norm in ["final_disease", "disease", "predicted_disease"]:
            rename_map[c] = "final_disease"

        elif c_norm == "disease_norm":
            rename_map[c] = "disease_norm"

        elif c_norm == "rrf_score":
            rename_map[c] = "rrf_score"

        elif c_norm == "dense_rank":
            rename_map[c] = "dense_rank"

        elif c_norm == "sparse_rank":
            rename_map[c] = "sparse_rank"

        elif c_norm == "rare_disease_name":
            rename_map[c] = "rare_disease_name"

    df = df.rename(columns=rename_map)

    required_cols = ["dataset", "patient_id", "final_rank", "final_disease"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"RRF file missing required columns: {missing_cols}\n"
            f"Current columns: {list(df.columns)}"
        )

    df["dataset"] = df["dataset"].apply(normalize_dataset)
    df["patient_id"] = df["patient_id"].apply(normalize_id)
    df["final_disease"] = df["final_disease"].astype(str).str.strip()
    df["final_rank"] = pd.to_numeric(df["final_rank"], errors="coerce")

    df = df.dropna(subset=["final_rank"]).copy()
    df["final_rank"] = df["final_rank"].astype(int)

    df["disease_match_key"] = df["final_disease"].apply(normalize_text)

    return df


# ============================================================
# Standardize claude_sections.xlsx
# ============================================================

def standardize_knowledge_columns(df):
    """
    claude_sections.xlsx should contain disease names and section columns.

    This function supports section columns like:

    clinical_presentation_section
    claude_clinical_presentation_section

    diagnostic_evaluation_section
    claude_diagnostic_evaluation_section

    subtype_variant_section
    claude_subtype_variant_section

    management_therapy_section
    claude_management_therapy_section
    """

    rename_map = {}

    for c in df.columns:
        c_norm = normalize_colname(c)

        # Disease name column
        if c_norm in [
            "disease",
            "disease_name",
            "rare_disease_name",
            "doc_id",
            "predicted_disease",
            "final_disease"
        ]:
            rename_map[c] = "disease"

        # Section columns
        elif (
            c_norm == "clinical_presentation"
            or c_norm == "clinical_presentation_section"
            or c_norm.endswith("_clinical_presentation_section")
        ):
            rename_map[c] = "clinical_presentation_section"

        elif (
            c_norm == "diagnostic_evaluation"
            or c_norm == "diagnostic_evaluation_section"
            or c_norm.endswith("_diagnostic_evaluation_section")
        ):
            rename_map[c] = "diagnostic_evaluation_section"

        elif (
            c_norm == "subtype_variant"
            or c_norm == "subtype_variant_section"
            or c_norm.endswith("_subtype_variant_section")
        ):
            rename_map[c] = "subtype_variant_section"

        elif (
            c_norm == "management_therapy"
            or c_norm == "management_therapy_section"
            or c_norm.endswith("_management_therapy_section")
        ):
            rename_map[c] = "management_therapy_section"

    df = df.rename(columns=rename_map)

    if "disease" not in df.columns:
        raise ValueError(
            "claude_sections.xlsx must contain a disease-name column.\n"
            f"Current columns after rename: {list(df.columns)}"
        )

    section_cols = [
        "clinical_presentation_section",
        "diagnostic_evaluation_section",
        "subtype_variant_section",
        "management_therapy_section"
    ]

    missing_sections = [c for c in section_cols if c not in df.columns]

    if missing_sections:
        raise ValueError(
            f"claude_sections.xlsx missing section columns after rename: {missing_sections}\n"
            f"Current columns after rename: {list(df.columns)}"
        )

    df["disease"] = df["disease"].astype(str).str.strip()
    df["disease_match_key"] = df["disease"].apply(normalize_text)

    for c in section_cols:
        df[c] = df[c].fillna("").astype(str)

    df = df.drop_duplicates(subset=["disease_match_key"], keep="first").copy()

    keep_cols = [
        "disease",
        "disease_match_key",
        "clinical_presentation_section",
        "diagnostic_evaluation_section",
        "subtype_variant_section",
        "management_therapy_section"
    ]

    return df[keep_cols]


# ============================================================
# Standardize combined_test_pts.xlsx
# ============================================================

def standardize_case_data_columns(df):
    """
    combined_test_pts.xlsx should contain:
    patient_id / person_id
    dataset
    case_data
    """

    rename_map = {}

    for c in df.columns:
        c_norm = normalize_colname(c)

        if c_norm in ["patient_id", "person_id", "pt_id"]:
            rename_map[c] = "patient_id"

        elif c_norm == "dataset":
            rename_map[c] = "dataset"

        elif c_norm in ["case_data", "case_text", "patient_case", "clinical_text"]:
            rename_map[c] = "case_data"

    df = df.rename(columns=rename_map)

    required_cols = ["patient_id", "dataset", "case_data"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"combined_test_pts.xlsx missing required columns: {missing_cols}\n"
            f"Current columns after rename: {list(df.columns)}"
        )

    df["patient_id"] = df["patient_id"].apply(normalize_id)
    df["dataset"] = df["dataset"].apply(normalize_dataset)
    df["case_data"] = df["case_data"].fillna("").astype(str)

    df = df[df["patient_id"].ne("") & df["dataset"].ne("")].copy()

    duplicated = df.duplicated(subset=["dataset", "patient_id"], keep=False).sum()

    if duplicated > 0:
        print(f"⚠️ duplicated patient_id + dataset rows in combined_test_pts.xlsx: {duplicated}")
        print("⚠️ Keeping first record for each patient_id + dataset.")

    df = df.drop_duplicates(subset=["dataset", "patient_id"], keep="first").copy()

    return df


def build_case_data_lookup_from_excel(case_data_file):
    if not os.path.exists(case_data_file):
        raise FileNotFoundError(f"case_data file not found: {case_data_file}")

    case_df = pd.read_excel(case_data_file)
    case_df = standardize_case_data_columns(case_df)

    lookup = {
        (row["dataset"], row["patient_id"]): row["case_data"]
        for _, row in case_df.iterrows()
    }

    return lookup, case_df


# ============================================================
# Load section knowledge from claude_sections.xlsx
# ============================================================

knowledge_df = pd.read_excel(knowledge_file)
knowledge_df = standardize_knowledge_columns(knowledge_df)

print(f"✅ Loaded section knowledge records: {len(knowledge_df)}")
print(f"✅ Unique section diseases: {knowledge_df['disease_match_key'].nunique()}")

display(knowledge_df.head())


# ============================================================
# Load case_data from combined_test_pts.xlsx
# ============================================================

case_data_lookup, case_df = build_case_data_lookup_from_excel(case_data_file)

print(f"✅ Loaded case_data records: {len(case_df)}")
print(f"✅ case_data lookup records: {len(case_data_lookup)}")

display(case_df.head())


# ============================================================
# Find RRF patient-level top10 files
# ============================================================

rrf_files = sorted(
    glob.glob(
        os.path.join(
            rrf_root,
            "*",
            "*",
            "*_rrf_final_top10.csv"
        )
    )
)

print(f"📂 RRF root: {rrf_root}")
print(f"📄 Found RRF patient files: {len(rrf_files)}")


# ============================================================
# Generate one JSON per patient
# ============================================================

summary_records = []

for i, rrf_file in enumerate(rrf_files, 1):

    print("-" * 100)
    print(f"📥 [{i}/{len(rrf_files)}] Reading RRF file:")
    print(rrf_file)

    try:
        rrf_df = pd.read_csv(rrf_file)
        rrf_df = standardize_rrf_columns(rrf_df)

    except Exception as e:
        print(f"❌ Failed to read RRF file: {e}")
        continue

    if rrf_df.empty:
        print("⚠️ Empty RRF file, skipping")
        continue

    rrf_df = rrf_df.sort_values("final_rank").head(top_n).copy()

    dataset = str(rrf_df["dataset"].iloc[0]).strip()
    patient_id = str(rrf_df["patient_id"].iloc[0]).strip()

    # Get case_data from combined_test_pts.xlsx
    case_data = case_data_lookup.get((dataset, patient_id), "")

    if case_data == "":
        print(f"⚠️ No case_data found for dataset={dataset}, patient_id={patient_id}")

    # Join RRF top10 diseases with section data from claude_sections.xlsx
    merged_df = rrf_df.merge(
        knowledge_df,
        on="disease_match_key",
        how="left",
        suffixes=("_rrf", "_section")
    )

    diseases_json = []

    for _, row in merged_df.iterrows():

        knowledge_found = not pd.isna(row.get("disease"))

        disease_record = {
            "rank": int(row["final_rank"]),
            "rrf_disease": row.get("final_disease"),
            "knowledge_found": bool(knowledge_found),

            # Section data from claude_sections.xlsx
            "clinical_presentation_section": "" if pd.isna(row.get("clinical_presentation_section")) else str(row.get("clinical_presentation_section")),
            "diagnostic_evaluation_section": "" if pd.isna(row.get("diagnostic_evaluation_section")) else str(row.get("diagnostic_evaluation_section")),
            "subtype_variant_section": "" if pd.isna(row.get("subtype_variant_section")) else str(row.get("subtype_variant_section")),
            "management_therapy_section": "" if pd.isna(row.get("management_therapy_section")) else str(row.get("management_therapy_section"))
        }

        diseases_json.append(disease_record)

    output_json = {
        "patient_id": patient_id,
        "dataset": dataset,
        "model": model_tag,
        "source": "RRF fusion final top10",
        "case_data": case_data,
        "diseases": diseases_json
    }

    pt_output_folder = os.path.join(
        json_output_root,
        safe_filename(dataset),
        safe_filename(patient_id)
    )

    os.makedirs(pt_output_folder, exist_ok=True)

    output_json_file = os.path.join(
        pt_output_folder,
        f"{safe_filename(patient_id)}_{safe_filename(dataset)}_{safe_filename(model_tag)}_rrf_top10_sections.json"
    )

    with open(output_json_file, "w", encoding="utf-8") as f:
        json.dump(output_json, f, ensure_ascii=False, indent=2)

    n_knowledge_found = sum(d["knowledge_found"] for d in diseases_json)

    n_nonempty_sections = 0

    for d in diseases_json:
        has_any_section = any([
            bool(d["clinical_presentation_section"].strip()),
            bool(d["diagnostic_evaluation_section"].strip()),
            bool(d["subtype_variant_section"].strip()),
            bool(d["management_therapy_section"].strip())
        ])

        if has_any_section:
            n_nonempty_sections += 1

    print(f"✅ Saved JSON: {output_json_file}")
    print(f"✅ case_data included: {bool(case_data.strip())}")
    print(f"✅ Section knowledge matched: {n_knowledge_found}/{len(diseases_json)}")
    print(f"✅ Diseases with non-empty sections: {n_nonempty_sections}/{len(diseases_json)}")

    summary_records.append({
        "model": model_tag,
        "dataset": dataset,
        "patient_id": patient_id,
        "has_case_data": int(bool(case_data.strip())),
        "case_data_length": len(case_data),
        "n_top_diseases": len(diseases_json),
        "n_knowledge_found": n_knowledge_found,
        "n_knowledge_missing": len(diseases_json) - n_knowledge_found,
        "n_diseases_with_nonempty_sections": n_nonempty_sections,
        "output_json_file": output_json_file,
        "source_rrf_file": rrf_file
    })


# ============================================================
# Save summary
# ============================================================

summary_df = pd.DataFrame(summary_records)

summary_file = os.path.join(
    json_output_root,
    f"{safe_filename(model_tag)}_rrf_top10_sections_json_summary.csv"
)

summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")

print("=" * 100)
print("🏁 Done generating RRF top10 section JSONs.")
print(f"📄 Summary saved: {summary_file}")

if not summary_df.empty:
    display(summary_df.head())

    print("\nSection knowledge match summary:")
    display(
        summary_df[[
            "n_top_diseases",
            "n_knowledge_found",
            "n_knowledge_missing",
            "n_diseases_with_nonempty_sections",
            "has_case_data",
            "case_data_length"
        ]].sum().to_frame("count")
    )

    print("\nPatients without case_data:")
    missing_case_df = summary_df[summary_df["has_case_data"] == 0].copy()
    display(missing_case_df[["dataset", "patient_id", "output_json_file"]])

    print("\nPatients with matched disease names but empty section content:")
    empty_section_df = summary_df[
        (summary_df["n_knowledge_found"] > 0) &
        (summary_df["n_diseases_with_nonempty_sections"] == 0)
    ].copy()

    display(empty_section_df[["dataset", "patient_id", "output_json_file"]])

✅ Loaded section knowledge records: 1320
✅ Unique section diseases: 1320


,disease,disease_match_key,clinical_presentation_section,diagnostic_evaluation_section,subtype_variant_section,management_therapy_section
0,48 XXYY Syndrome,48 xxyy syndrome,Core Signs & Symptoms:**\n\n*Early Development...,Clinical Criteria:** \nDiagnosis is often made...,No formal subtypes reported. \nStandard karyot...,**First-Line Treatments:** \nIf hypogonadism i...
1,AEC Syndrome,aec syndrome,Core Signs & Symptoms:**\n\n*Early/Neonatal Pr...,Clinical Criteria:** \nClinical features can b...,The Rapp-Hodgkin syndrome is not a separate di...,**First-Line Treatments:**\n- \nEfforts to pre...
2,ASAH1-Related Disorders,asah1 related disorders,"Core Signs & Symptoms:** \nThe classic form, s...",Clinical Criteria:** \nThe diagnosis of an ASA...,"Historically, Farber disease has been broken d...",**First-Line Treatments:** \nThere is presentl...
3,Abetalipoproteinemia,abetalipoproteinemia,Core Signs & Symptoms**: \n\n*Early Infancy Si...,Clinical Criteria**: \nNo formal clinical diag...,"No formal subtypes reported. However, \nmany A...",**First-Line Treatments**: \nTreatment include...
4,Achondroplasia,achondroplasia,Core Signs & Symptoms:**\n\n**Physical Charact...,Clinical Criteria:** \nBoth the clinical and r...,Severe achondroplasia with developmental delay...,**First-Line Treatments:**\n\n**Vosoritide (Vo...


✅ Loaded case_data records: 9290
✅ case_data lookup records: 9290


,patient_id,case_data,dataset
0,9425591-1,A 66-year-old Caucasian man was referred to ou...,pubmed
1,9453449-1,A 7-year-old boy was brought to the local hosp...,pubmed
2,9525054-1,The patient is a 66-year-old Caucasian male wh...,pubmed
3,9570535-3,"On October 2021, a 17-years-old boy referred t...",pubmed
4,9593512-1,A 60-year-old female patient presented with dy...,pubmed


📂 RRF root: /Users/sandersu1/downloads/fusion_dense_sparse_rrf_local/claude
📄 Found RRF patient files: 10
----------------------------------------------------------------------------------------------------
📥 [1/10] Reading RRF file:
/Users/sandersu1/downloads/fusion_dense_sparse_rrf_local/claude/pubmed/6142631-1/6142631-1_pubmed_claude_rrf_final_top10.csv
✅ Saved JSON: /Users/sandersu1/downloads/rrf_top10_patient_jsons/claude/pubmed/6142631-1/6142631-1_pubmed_claude_rrf_top10_sections.json
✅ case_data included: True
✅ Section knowledge matched: 10/10
✅ Diseases with non-empty sections: 10/10
----------------------------------------------------------------------------------------------------
📥 [2/10] Reading RRF file:
/Users/sandersu1/downloads/fusion_dense_sparse_rrf_local/claude/pubmed/6219329-1/6219329-1_pubmed_claude_rrf_final_top10.csv
✅ Saved JSON: /Users/sandersu1/downloads/rrf_top10_patient_jsons/claude/pubmed/6219329-1/6219329-1_pubmed_claude_rrf_top10_sections.json
✅ case_dat

...

,count
n_top_diseases,100
n_knowledge_found,100
n_knowledge_missing,0
n_diseases_with_nonempty_sections,100
has_case_data,10
case_data_length,25399



Patients without case_data:


,dataset,patient_id,output_json_file



Patients with matched disease names but empty section content:


,dataset,patient_id,output_json_file


## Run Qwen3-8B Reranker with LLM generated disease knowledge sections

In [ ]:

# ============================================================
# Settings
# ============================================================

# Input folder we just generated:

input_root = os.path.join(
    base_dir,
    "rrf_top10_patient_jsons",
    model_tag
)

# Output folder
output_root = os.path.join(
    base_dir,
    "qwen_reranker_rrf_top10_output",
    model_tag
)

os.makedirs(output_root, exist_ok=True)

# Hugging Face model repo.
# First run downloads it. Later runs reuse cache.
model_repo = "Qwen/Qwen3-Reranker-8B"

# Local cache folder
hf_cache_dir = os.path.join(
    base_dir,
    "hf_model_cache"
)

os.makedirs(hf_cache_dir, exist_ok=True)

max_length = 8192
field_name = "similarity_all_combined"


# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    x = str(x)
    x = x.replace("/", "_").replace("\\", "_").replace(" ", "_")
    return x


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")

    #if torch.backends.mps.is_available():
    #    return torch.device("mps")

    return torch.device("cpu")


# ============================================================
# Load Qwen3 reranker model
# ============================================================

def load_reranker_model(model_repo, hf_cache_dir=None):
    device = get_device()

    print(f"🔄 Loading model: {model_repo}")
    print(f"📦 Cache dir: {hf_cache_dir}")
    print(f"🖥️ Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_repo,
        padding_side="left",
        use_fast=True,
        cache_dir=hf_cache_dir
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    #if device.type in ["cuda", "mps"]:
    if device.type in ["cuda"]:
        torch_dtype = torch.float16
    else:
        torch_dtype = torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        model_repo,
        torch_dtype=torch_dtype,
        cache_dir=hf_cache_dir,
        low_cpu_mem_usage=True
    )

    model = model.to(device)
    model.eval()

    prefix = (
        "<|im_start|>system\n"
        "Judge whether the Document meets the requirements based on the Query and the Instruct provided. "
        "Note that the answer can only be \"yes\" or \"no\"."
        "<|im_end|>\n"
        "<|im_start|>user\n"
    )

    suffix = (
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        "<think>\n\n</think>\n\n"
    )

    prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
    suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)

    token_true_id = tokenizer.convert_tokens_to_ids("yes")
    token_false_id = tokenizer.convert_tokens_to_ids("no")

    if token_true_id is None or token_true_id == tokenizer.unk_token_id:
        raise ValueError("Cannot find token id for 'yes'.")

    if token_false_id is None or token_false_id == tokenizer.unk_token_id:
        raise ValueError("Cannot find token id for 'no'.")

    return (
        model,
        tokenizer,
        prefix_tokens,
        suffix_tokens,
        token_true_id,
        token_false_id,
        device
    )


# ============================================================
# Format input
# ============================================================

def format_instruction(instruction, query, doc):
    return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}"


def process_inputs(
    pairs,
    tokenizer,
    prefix_tokens,
    suffix_tokens,
    device,
    max_length=8192
):
    prefix_ids = torch.tensor(prefix_tokens, dtype=torch.long)
    suffix_ids = torch.tensor(suffix_tokens, dtype=torch.long)

    encoded = tokenizer(
        pairs,
        padding=False,
        truncation=True,
        return_tensors=None,
        max_length=max_length - len(prefix_tokens) - len(suffix_tokens)
    )

    input_ids_list = []

    for ids in encoded["input_ids"]:
        seq = torch.tensor(ids, dtype=torch.long)
        full_seq = torch.cat([prefix_ids, seq, suffix_ids])
        full_seq = full_seq[:max_length]
        input_ids_list.append(full_seq)

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    attention_mask = input_ids.ne(tokenizer.pad_token_id).long()

    inputs = {
        "input_ids": input_ids.to(device),
        "attention_mask": attention_mask.to(device)
    }

    return inputs


# ============================================================
# Compute rerank scores
# ============================================================

@torch.no_grad()
def compute_rerank_scores(inputs, model, token_true_id, token_false_id):
    outputs = model(**inputs)
    logits = outputs.logits

    # Important:
    # Because we pad sequences, the last column may be PAD.
    # So we score the final real token for each row.
    attention_mask = inputs["attention_mask"]
    last_token_indices = attention_mask.sum(dim=1) - 1

    batch_indices = torch.arange(
        logits.size(0),
        device=logits.device
    )

    final_logits = logits[batch_indices, last_token_indices, :]

    true_logits = final_logits[:, token_true_id]
    false_logits = final_logits[:, token_false_id]

    combined = torch.stack([false_logits, true_logits], dim=1)
    log_probs = torch.nn.functional.log_softmax(combined, dim=1)

    scores = log_probs[:, 1].exp().detach().cpu().tolist()

    return scores


# ============================================================
# Disease document builder
# ============================================================

def build_disease_doc(d):
    ref_disease = d.get("rrf_disease", "")

    doc = (
        f"{ref_disease}\n"
        f"### Clinical Presentation:\n{d.get('clinical_presentation_section', '')}\n"
        f"### Diagnostic Evaluation:\n{d.get('diagnostic_evaluation_section', '')}\n"
        f"### Subtype Variant:\n{d.get('subtype_variant_section', '')}\n"
        f"### Management Therapy:\n{d.get('management_therapy_section', '')}"
    )

    return doc.strip()


# ============================================================
# Rerank diseases
# ============================================================

def rerank_by_qwen(
    case_text,
    diseases,
    model,
    tokenizer,
    prefix_tokens,
    suffix_tokens,
    token_true_id,
    token_false_id,
    device
):
    instruction = (
        "Given a patient case description, determine whether the document describes "
        "a disease that matches or explains the patient's condition."
    )

    pairs = [
        format_instruction(
            instruction,
            case_text,
            build_disease_doc(d)
        )
        for d in diseases
    ]

    inputs = process_inputs(
        pairs=pairs,
        tokenizer=tokenizer,
        prefix_tokens=prefix_tokens,
        suffix_tokens=suffix_tokens,
        device=device,
        max_length=max_length
    )

    scores = compute_rerank_scores(
        inputs=inputs,
        model=model,
        token_true_id=token_true_id,
        token_false_id=token_false_id
    )

    detailed_scores = []

    for d, s in zip(diseases, scores):
        ref_disease = d.get("rrf_disease", "")

        record = {
            "original_rrf_rank": d.get("rank", None),
            "ref_disease": ref_disease,
            field_name: round(float(s), 4)
        }

        detailed_scores.append(record)

    detailed_scores = sorted(
        detailed_scores,
        key=lambda x: x[field_name],
        reverse=True
    )

    for i, d in enumerate(detailed_scores, 1):
        d["qwen_rerank_position"] = i

    return detailed_scores


# ============================================================
# Process one JSON file
# ============================================================

def process_json_file(
    json_path,
    model,
    tokenizer,
    prefix_tokens,
    suffix_tokens,
    token_true_id,
    token_false_id,
    device,
    output_root
):
    with open(json_path, "r", encoding="utf-8") as f:
        patient_entry = json.load(f)

    patient_id = str(patient_entry.get("patient_id", "")).strip()
    dataset = str(patient_entry.get("dataset", "")).strip()
    case_text = patient_entry.get("case_data", "")
    diseases = patient_entry.get("diseases", [])

    if patient_id == "":
        raise ValueError(f"Missing patient_id in {json_path}")

    if dataset == "":
        dataset = "unknown_dataset"

    if not case_text or str(case_text).strip() == "":
        raise ValueError(
            f"Missing case_data for patient_id={patient_id}, dataset={dataset}"
        )

    if not diseases:
        raise ValueError(
            f"No diseases found for patient_id={patient_id}, dataset={dataset}"
        )

    output_dataset_dir = os.path.join(
        output_root,
        safe_filename(dataset)
    )

    os.makedirs(output_dataset_dir, exist_ok=True)

    base_name = os.path.splitext(os.path.basename(json_path))[0]

    output_txt = os.path.join(
        output_dataset_dir,
        f"{base_name}_qwen_reranked.txt"
    )

    if os.path.exists(output_txt) and os.path.getsize(output_txt) > 0:
        print(f"⏭️ Skipping already processed: {output_txt}")
        return

    detailed_scores = rerank_by_qwen(
        case_text=case_text,
        diseases=diseases,
        model=model,
        tokenizer=tokenizer,
        prefix_tokens=prefix_tokens,
        suffix_tokens=suffix_tokens,
        token_true_id=token_true_id,
        token_false_id=token_false_id,
        device=device
    )

    output_data = {
        "patient_id": patient_id,
        "dataset": dataset,
        "model": model_tag,
        "source_json": json_path,
        "case_data": case_text,
        "detailed_scores": detailed_scores
    }

    with open(output_txt, "w", encoding="utf-8") as out_f:
        json.dump(
            output_data,
            out_f,
            ensure_ascii=False,
            indent=2
        )

    print(f"✅ Reranked: {output_txt}")


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    print(f"📂 Input root: {input_root}")
    print(f"📂 Output root: {output_root}")

    if not os.path.exists(input_root):
        raise FileNotFoundError(f"Input folder not found: {input_root}")

    model, tokenizer, prefix_tokens, suffix_tokens, token_true_id, token_false_id, device = load_reranker_model(
        model_repo=model_repo,
        hf_cache_dir=hf_cache_dir
    )

    all_json_files = sorted(
        glob.glob(
            os.path.join(input_root, "**", "*.json"),
            recursive=True
        )
    )

    print(f"📄 Found JSON files: {len(all_json_files)}")

    for json_path in tqdm(
        all_json_files,
        desc="📊 Reranking RRF top10 JSONs with Qwen3",
        unit="file"
    ):
        try:
            process_json_file(
                json_path=json_path,
                model=model,
                tokenizer=tokenizer,
                prefix_tokens=prefix_tokens,
                suffix_tokens=suffix_tokens,
                token_true_id=token_true_id,
                token_false_id=token_false_id,
                device=device,
                output_root=output_root
            )

        except Exception as e:
            print(f"❌ Error in {json_path}: {e}")

            error_dir = os.path.join(output_root, "_errors")
            os.makedirs(error_dir, exist_ok=True)

            base_name = os.path.splitext(os.path.basename(json_path))[0]
            error_file = os.path.join(
                error_dir,
                f"{base_name}_error.json"
            )

            with open(error_file, "w", encoding="utf-8") as f:
                json.dump(
                    {
                        "source_json": json_path,
                        "error": str(e)
                    },
                    f,
                    ensure_ascii=False,
                    indent=2
                )

    print("🏁 Done.")

📂 Input root: /Users/sandersu1/downloads/rrf_top10_patient_jsons/claude
📂 Output root: /Users/sandersu1/downloads/qwen_reranker_rrf_top10_output/claude
🔄 Loading model: Qwen/Qwen3-Reranker-8B
📦 Cache dir: /Users/sandersu1/downloads/hf_model_cache
🖥️ Device: cpu


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

📄 Found JSON files: 10


📊 Reranking RRF top10 JSONs with Qwen3:  10%| | 1/10 [05:18<47:42, 318.09s/file

✅ Reranked: /Users/sandersu1/downloads/qwen_reranker_rrf_top10_output/claude/pubmed/6142631-1_pubmed_claude_rrf_top10_sections_qwen_reranked.txt


...

✅ Reranked: /Users/sandersu1/downloads/qwen_reranker_rrf_top10_output/claude/pubmed/6886627-1_pubmed_claude_rrf_top10_sections_qwen_reranked.txt


📊 Reranking RRF top10 JSONs with Qwen3:  80%|▊| 8/10 [36:11<08:57, 268.89s/file

✅ Reranked: /Users/sandersu1/downloads/qwen_reranker_rrf_top10_output/claude/pubmed/6935327-1_pubmed_claude_rrf_top10_sections_qwen_reranked.txt


📊 Reranking RRF top10 JSONs with Qwen3:  90%|▉| 9/10 [41:02<04:35, 275.62s/file

✅ Reranked: /Users/sandersu1/downloads/qwen_reranker_rrf_top10_output/claude/pubmed/7007742-1_pubmed_claude_rrf_top10_sections_qwen_reranked.txt
